# Configuration

In [1]:
from pathlib import Path
from hashlib import sha256
import json
import re

import joblib
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from IPython.display import display


# Project paths

def locate_project_root():
    """Find the project root from the repository or notebooks directory."""
    here = Path.cwd().resolve()

    for candidate in (here, *here.parents):
        if (
            (candidate / "notebooks").is_dir()
            and (candidate / "requirements.txt").is_file()
        ):
            return candidate

    return here.parent if here.name == "notebooks" else here


ROOT = locate_project_root()
DATA_DIR = ROOT / "data"

DEVELOPMENT_DIR = DATA_DIR / "development"
SUPPORTING_DIR = DATA_DIR / "supporting"
SEALED_DIR = DATA_DIR / "sealed"

ORIGINAL_MODEL_PATH = (
    ROOT / "models" / "exp10b_model.joblib"
)

ORIGINAL_METADATA_PATH = (
    ROOT / "models" / "exp10b_model_metadata.json"
)

OUTPUT_DIR = ROOT / "outputs" / "stage4"


# Reproducibility

SEED = 42
np.random.seed(SEED)


# Challenge rules

STARTING_BALANCE = 5_000.0

MAX_DRAWDOWN_RATE = 0.04

DRAWDOWN_LIMIT_AMOUNT = (
    STARTING_BALANCE * MAX_DRAWDOWN_RATE
)

REALIZED_DRAWDOWN_BOUNDARY = (
    -DRAWDOWN_LIMIT_AMOUNT
)

PROFIT_TARGET_RATE = 0.08

PROFIT_TARGET_AMOUNT = (
    STARTING_BALANCE * PROFIT_TARGET_RATE
)

# C22 confirmed that everyone in a campaign shares
# the same 24-hour window: 9 AM to 9 AM Singapore time.
CHALLENGE_TIMEZONE = "Asia/Singapore"
CHALLENGE_START_HOUR = 9
CHALLENGE_DURATION_HOURS = 24

# Do not infer official dates from first/last observed trades.
# Desperate pace remains disabled until official dates
# are available and its cutoff has been developed.
CAMPAIGN_DEADLINES_UTC = {}

# Campaign 33 had a delayed operational cutoff.
# Apply the official 9 AM SGT cutoff when its date is verified.
CAMPAIGNS_WITH_CUTOFF_EXCEPTIONS = (33,)


# Campaign splits

TRAIN_START = 33
TRAIN_END = 52

SUPPORT_START = 53
SUPPORT_END = 66

SEALED_CAMPAIGNS = tuple(
    range(67, 83)
)

# User confirmed that campaigns 67-82 were downloaded
# but have not been evaluated.
EXPOSED_CAMPAIGNS = ()


# Person identification

# C22 confirmed email as the intended person identifier.
# First join User Trades to User Data within each campaign
# using campaign_id + account_id.
PERSON_ID_METHOD = "normalised_email"

# Flag unresolved accounts rather than linking account IDs
# across campaigns or dropping their trades silently.
MISSING_EMAIL_POLICY = "campaign_specific_fallback"


# Frozen model specification

FEATURE_COLUMNS = [
    "post_loss_entry",
    "post_loss_reentry_gap_minutes",
    "post_loss_amount_ratio",
    "past_trade_open_count",
    "elapsed_active_hours",
    "past_trades_opened_per_active_hour",
    "trades_opened_past_30_minutes",
    "minutes_since_previous_trade_open",
    "past_median_trade_open_gap_minutes",
    "trade_open_gap_to_past_median_ratio",
    "minutes_since_previous_idea_start",
    "past_median_idea_start_gap_minutes",
    "idea_start_gap_to_past_median_ratio",
    "realized_distance_to_drawdown_limit",
    "historical_median_amount_before_trade",
    "current_to_historical_median_amount_ratio",
    "past_amount_cv",
    "past_completed_idea_count",
    "past_idea_win_rate",
    "past_mean_win_profit_per_lot",
    "past_mean_loss_abs_profit_per_lot",
    "past_payoff_ratio",
]

MODEL_PARAMETERS = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "num_leaves": 15,
    "random_state": SEED,
    "verbosity": -1,
}

# Quantile-based threshold for walk-forward refitting.
QUANTILE = 0.90

# Absolute threshold used only by the original
# Stage 3 continuity strategy.
ORIGINAL_THRESHOLD = 85.102037


# Stand-down filter

# Candidate cutoffs only. Retain a rule if its
# development-set lot-weighted RP/lot is negative
# and its 95% CI lies entirely below zero.
FILTER_CUTOFFS = {
    "size_up_ratio": 2.0,
    "hot_streak_completed_wins": 3,
    "near_target_remaining_profit": None,
    "desperate_pace_profit_per_hour": None,
}


# Evaluation settings

BOOTSTRAP_ITERATIONS = 2_000
CONFIDENCE_LEVEL = 0.95

# Registered Stage 4 strategies

REGISTERED_STRATEGIES = {
    "walk_forward_gate": (
        "Expanding-window LightGBM refit; "
        "training-score 90th-percentile gate; PRIMARY"
    ),
    "original_continuity": (
        "Original campaigns 33-52 fitted model; "
        "fixed numerical threshold 85.102037"
    ),
}

# Holm correction across the two registered tests.
MULTIPLE_TESTING_METHOD = "holm"

# Primary pass criteria.
MIN_POSITIVE_CAMPAIGN_SHARE = 0.60


# Sealed evaluation safeguards

# Both switches must be True before the final evaluation.
RUN_SEALED_EVALUATION = True
PROTOCOL_APPROVED = True


# Configuration summary

print("=" * 60)
print("STAGE 4 CONFIGURATION")
print("=" * 60)

print("Project root:", ROOT)
print("Development directory:", DEVELOPMENT_DIR)
print("Supporting directory:", SUPPORTING_DIR)
print("Sealed directory:", SEALED_DIR)
print("Output directory:", OUTPUT_DIR)

print("\nChallenge configuration:")
print(f"Starting balance: ${STARTING_BALANCE:,.2f}")
print(f"Profit target: ${PROFIT_TARGET_AMOUNT:,.2f}")
print(
    "Challenge window:",
    "9 AM to 9 AM SGT",
)

print("\nModel configuration:")
print("Features:", len(FEATURE_COLUMNS))
print("Walk-forward quantile:", QUANTILE)
print("Original threshold:", ORIGINAL_THRESHOLD)

print("\nPerson identification:", PERSON_ID_METHOD)

print("\nSealed campaigns:", SEALED_CAMPAIGNS)
print("Exposed campaigns:", EXPOSED_CAMPAIGNS)

print(
    "Sealed evaluation:",
    (
        "ENABLED"
        if RUN_SEALED_EVALUATION and PROTOCOL_APPROVED
        else "LOCKED"
    ),
)

print("Configuration ready.")

ModuleNotFoundError: No module named 'lightgbm'

## Input paths

In [ ]:
required_directories = [
    ("development trades", DEVELOPMENT_DIR / "User Trades"),
    ("development user data", DEVELOPMENT_DIR / "User Data"),
    ("supporting trades", SUPPORTING_DIR / "User Trades"),
    ("supporting user data", SUPPORTING_DIR / "User Data"),
]

for label, path in required_directories:
    if not path.is_dir():
        raise FileNotFoundError(f"Missing {label}: {path}")

    print(f"Found {label}: {path}")

print("Development and supporting directories checked.")
print("No sealed files opened.")

Found development trades: /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/data/development/User Trades
Found development user data: /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/data/development/User Data
Found supporting trades: /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/data/supporting/User Trades
Found supporting user data: /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/data/supporting/User Data
Development and supporting directories checked.
No sealed files opened.


# Data preparation

## Trade schema and file loading

In [ ]:
FILENAME_PATTERN = re.compile(
    r"Campaign[_\s]*(\d+)"
    r"[_\s]*Data[_\s]*"
    r"(\d{1,2}[_\s]*[A-Za-z]+[_\s]*\d{4})"
    r".*?"
    r"(Traders[\s_]*only|XAUUSD[\s_]*only)",
    re.IGNORECASE,
)


TRADE_ID_COLUMNS = [
    "account_id",
    "close_trade_id",
    "position_id",
    "close_order_id",
    "open_order_id",
    "user_group_id",
]


TRADE_DATETIME_COLUMNS = [
    "open_date_time",
    "close_date_time",
]


TRADE_NUMERIC_COLUMNS = [
    "lot_size",
    "duration_sec",
    "profit",
    "reverse_profit",
    "net_profit",
    "commission",
    "swap",
    "amount",
    "open_price",
    "close_price",
    "sl_price",
    "tp_price",
    "open_trade_cross_price",
    "close_trade_cross_price",
]


TRADE_COLUMN_ALIASES = {
    "account_id": "account_id",
    "accountid": "account_id",

    "close_trade_id": "close_trade_id",
    "closetradeid": "close_trade_id",

    "position_id": "position_id",
    "positionid": "position_id",

    "close_order_id": "close_order_id",
    "closeorderid": "close_order_id",

    "open_order_id": "open_order_id",
    "openorderid": "open_order_id",

    "user_group_id": "user_group_id",
    "usergroupid": "user_group_id",

    "open_date_time": "open_date_time",
    "opendatetime": "open_date_time",

    "close_date_time": "close_date_time",
    "closedatetime": "close_date_time",

    "lot_size": "lot_size",
    "lotsize": "lot_size",

    "duration_sec": "duration_sec",
    "durationsec": "duration_sec",

    "profit": "profit",

    "reverse_profit": "reverse_profit",
    "reverseprofit": "reverse_profit",

    "net_profit": "net_profit",
    "netprofit": "net_profit",

    "commission": "commission",
    "swap": "swap",
    "amount": "amount",

    "open_price": "open_price",
    "openprice": "open_price",

    "close_price": "close_price",
    "closeprice": "close_price",

    "sl_price": "sl_price",
    "slprice": "sl_price",

    "tp_price": "tp_price",
    "tpprice": "tp_price",

    "open_trade_cross_price":
        "open_trade_cross_price",
    "opentradecrossprice":
        "open_trade_cross_price",

    "close_trade_cross_price":
        "close_trade_cross_price",
    "closetradecrossprice":
        "close_trade_cross_price",

    "side": "side",
    "currency": "currency",

    "campaign_id": "source_campaign_id",
    "campaignid": "source_campaign_id",
}


In [ ]:
def normalize_column_name(
    column,
):
    """Convert a raw column name to lowercase snake case."""

    column_name = str(
        column
    ).strip()

    column_name = re.sub(
        r"([A-Z]+)([A-Z][a-z])",
        r"\1_\2",
        column_name,
    )

    column_name = re.sub(
        r"([a-z0-9])([A-Z])",
        r"\1_\2",
        column_name,
    )

    column_name = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        column_name,
    )

    column_name = re.sub(
        r"_+",
        "_",
        column_name,
    )

    return (
        column_name
        .strip("_")
        .lower()
    )


In [ ]:
def parse_campaign_date(
    date_string,
):
    """Parse the campaign date encoded in a filename."""

    normalized_date_string = re.sub(
        r"[_\s]+",
        " ",
        date_string.strip(),
    )

    for date_format in [
        "%d %b %Y",
        "%d %B %Y",
    ]:
        parsed_date = pd.to_datetime(
            normalized_date_string,
            format=date_format,
            errors="coerce",
        )

        if pd.notna(
            parsed_date
        ):
            return parsed_date

    return pd.NaT


In [ ]:
def parse_filename(
    path,
):
    """Extract campaign metadata from a raw trade filename."""

    path = Path(
        path
    )

    filename = (
        path.stem
    )

    match = (
        FILENAME_PATTERN
        .search(
            filename
        )
    )

    if match:
        return {
            "campaign_id":
                int(
                    match.group(1)
                ),
            "campaign_date":
                parse_campaign_date(
                    match.group(2)
                ),
        }

    campaign_match = re.search(
        r"Campaign[_\s]*(\d+)",
        filename,
        re.IGNORECASE,
    )

    campaign_id = (
        int(
            campaign_match.group(1)
        )
        if campaign_match
        else None
    )

    return {
        "campaign_id":
            campaign_id,
        "campaign_date":
            pd.NaT,
    }


In [ ]:
def load_trade_file(
    path,
):
    """Load and standardize one raw campaign trade file."""

    path = Path(
        path
    )

    metadata = (
        parse_filename(
            path
        )
    )

    if (
        metadata[
            "campaign_id"
        ]
        is None
    ):
        raise ValueError(
            "Could not determine campaign ID "
            f"from {path.name}."
        )

    if (
        path.suffix.lower()
        == ".csv"
    ):
        df = pd.read_csv(
            path,
            low_memory=False,
        )

    elif (
        path.suffix.lower()
        == ".xlsx"
    ):
        df = pd.read_excel(
            path
        )

    else:
        raise ValueError(
            f"Unsupported file type: "
            f"{path.suffix}"
        )

    normalized_columns = {
        column:
            normalize_column_name(
                column
            )
        for column in df.columns
    }

    df = df.rename(
        columns=normalized_columns
    )

    df = df.rename(
        columns=TRADE_COLUMN_ALIASES
    )

    df.insert(
        0,
        "source_row_number",
        np.arange(
            2,
            len(df) + 2,
        ),
    )

    required_columns = {
        "account_id",
        "open_date_time",
        "amount",
        "side",
    }

    missing_required = (
        required_columns
        - set(
            df.columns
        )
    )

    if missing_required:
        raise ValueError(
            f"{path.name} is missing "
            "required columns: "
            f"{sorted(missing_required)}"
        )

    if "close_date_time" not in df.columns:
        df["close_date_time"] = pd.NaT
    if "net_profit" not in df.columns:
        df["net_profit"] = np.nan
    if "reverse_profit" not in df.columns:
        df["reverse_profit"] = np.nan

    header_echo_mask = (
        df[
            "account_id"
        ]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin(
            {
                "account_id",
                "accountid",
                "account",
            }
        )
        .fillna(False)
        |
        df[
            "open_date_time"
        ]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin(
            {
                "open_date_time",
                "opendatetime",
            }
        )
        .fillna(False)
    )

    df = (
        df.loc[
            ~header_echo_mask
        ]
        .copy()
    )

    for column in (
        TRADE_ID_COLUMNS
    ):
        if (
            column
            in df.columns
        ):
            df[
                column
            ] = (
                df[column]
                .astype("string")
                .str.strip()
                .replace(
                    {
                        "":
                            pd.NA,
                        "nan":
                            pd.NA,
                        "None":
                            pd.NA,
                    }
                )
            )

    for column in (
        TRADE_NUMERIC_COLUMNS
    ):
        if (
            column
            in df.columns
        ):
            df[
                column
            ] = pd.to_numeric(
                df[column],
                errors="coerce",
            )

    for column in (
        TRADE_DATETIME_COLUMNS
    ):
        if (
            column
            in df.columns
        ):
            df[
                column
            ] = pd.to_datetime(
                df[column],
                errors="coerce",
                utc=True,
            )

    df[
        "side"
    ] = (
        df[
            "side"
        ]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    if (
        "currency"
        in df.columns
    ):
        df[
            "currency"
        ] = (
            df[
                "currency"
            ]
            .astype("string")
            .str.strip()
            .str.upper()
        )

    df[
        "campaign_id"
    ] = metadata[
        "campaign_id"
    ]

    df[
        "campaign_date"
    ] = metadata[
        "campaign_date"
    ]

    df[
        "source_file"
    ] = path.name

    df[
        "source_path"
    ] = str(
        path
    )

    return (
        df.reset_index(
            drop=True
        )
    )


In [ ]:
def load_trade_directory(
    directory,
    is_unseen,
):
    """Load every supported trade file in a directory."""

    directory = Path(
        directory
    )

    dataset_label = (
        "UNSEEN"
        if is_unseen
        else "HISTORICAL"
    )

    print(
        f"\nScanning {dataset_label.lower()} "
        f"trade directory:"
    )
    print(
        f"  {directory}"
    )

    if not directory.exists():
        raise FileNotFoundError(
            f"Trade directory not found: "
            f"{directory.resolve()}"
        )

    paths = sorted(
        [
            path
            for path in directory.rglob("*")
            if (
                path.is_file()
                and path.suffix.lower()
                in {
                    ".csv",
                    ".xlsx",
                }
            )
        ]
    )

    if not paths:
        raise ValueError(
            "No CSV or XLSX trade files "
            f"found in {directory}."
        )

    print(
        f"Found {len(paths)} "
        f"{dataset_label.lower()} files."
    )

    frames = []

    for file_number, path in enumerate(
        paths,
        start=1,
    ):
        print(
            f"  [{file_number}/{len(paths)}] "
            f"Processing: {path.name}"
        )

        frame = (
            load_trade_file(
                path
            )
        )

        frame[
            "_is_unseen"
        ] = bool(
            is_unseen
        )

        frames.append(
            frame
        )

    combined = pd.concat(
        frames,
        ignore_index=True,
    )

    print(
        f"Loaded {len(combined):,} "
        f"{dataset_label.lower()} trades."
    )

    return combined


## User-data identity audit

In [ ]:
def campaign_from_path(path):
    """Extract a campaign ID from a filename or parent directory."""
    path = Path(path)

    for part in (path.stem, *(parent.name for parent in path.parents)):
        match = re.search(
            r"(?:Campaign[_\s-]*|\bC)(\d{2,3})(?!\d)",
            part,
            re.IGNORECASE,
        )

        if match:
            return int(match.group(1))

    raise ValueError(
        f"Cannot determine campaign ID from {path}."
    )


def normalise_account(values):
    """Normalise account IDs for campaign-specific joins."""
    result = (
        values.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    return result.mask(
        result.str.lower().isin(
            ["", "nan", "none", "null", "<na>"]
        )
    )


def normalise_email(values):
    """Normalise emails while preserving missing values."""
    result = (
        values.astype("string")
        .str.strip()
        .str.lower()
    )

    return result.mask(
        result.isin(
            ["", "nan", "none", "null", "<na>"]
        )
    )


def load_user_directory(directory):
    """Load and normalise User Data without constructing identities."""
    directory = Path(directory)

    if not directory.is_dir():
        raise FileNotFoundError(
            f"Missing User Data directory: {directory}"
        )

    paths = sorted(
        path
        for path in directory.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower() in {".csv", ".xlsx"}
            and not path.name.startswith("~$")
        )
    )

    if not paths:
        raise ValueError(
            f"No User Data files found in {directory}"
        )

    frames = []

    print(f"Loading {len(paths)} User Data files from {directory}...")

    for index, path in enumerate(paths, start=1):

        # Load file.
        if path.suffix.lower() == ".csv":
            frame = pd.read_csv(
                path,
                dtype="string",
                low_memory=False,
            )
        else:
            frame = pd.read_excel(
                path,
                dtype="string",
            )

        # Normalise column names.
        frame.columns = [
            normalize_column_name(column)
            for column in frame.columns
        ]

        # Handle known User Data aliases.
        frame = frame.rename(
            columns={
                "account": "account_id",
                "accountid": "account_id",
                "emailaddress": "email",
                "ipaddress": "ip_address",
            }
        )

        # Reject ambiguous columns.
        if frame.columns.duplicated().any():
            duplicated = (
                frame.columns[
                    frame.columns.duplicated()
                ].tolist()
            )

            raise ValueError(
                f"{path.name}: duplicated columns "
                f"after normalisation: {duplicated}"
            )

        required_columns = {
            "account_id",
            "email",
        }

        missing_columns = (
            required_columns - set(frame.columns)
        )

        if missing_columns:
            raise ValueError(
                f"{path.name}: missing required columns "
                f"{sorted(missing_columns)}. "
                f"Available columns: {frame.columns.tolist()}"
            )

        # Retain only fields needed for identity mapping.
        frame = frame[
            ["account_id", "email"]
        ].copy()

        frame["account_id"] = normalise_account(
            frame["account_id"]
        )

        frame["email"] = normalise_email(
            frame["email"]
        )

        frame["campaign_id"] = campaign_from_path(
            path
        )

        frames.append(frame)

        print(
            f"  [{index}/{len(paths)}] "
            f"Campaign {frame['campaign_id'].iloc[0]}: "
            f"{len(frame):,} user records"
        )

    users = pd.concat(
        frames,
        ignore_index=True,
    )

    print(
        f"Total User Data rows loaded: {len(users):,}"
    )

    return users

In [ ]:
def validate_campaign_range(
    data,
    first_campaign,
    last_campaign,
    label,
):
    """Verify that a dataset contains exactly the expected campaigns."""
    expected = set(
        range(first_campaign, last_campaign + 1)
    )

    actual = set(
        data["campaign_id"]
        .dropna()
        .astype(int)
        .unique()
    )

    if actual != expected:
        raise ValueError(
            f"{label}: "
            f"Missing campaigns: {sorted(expected - actual)}. "
            f"Unexpected campaigns: {sorted(actual - expected)}."
        )

    print(
        f"{label}: {len(actual)} campaigns verified "
        f"({first_campaign}–{last_campaign})."
    )


print("=" * 60)
print("LOADING HISTORICAL USER DATA")
print("=" * 60)

# Development: 33–52
development_users = load_user_directory(
    DEVELOPMENT_DIR / "User Data"
)

# Supporting: 53–66
supporting_users = load_user_directory(
    SUPPORTING_DIR / "User Data"
)

validate_campaign_range(
    development_users,
    TRAIN_START,
    TRAIN_END,
    "Development User Data",
)

validate_campaign_range(
    supporting_users,
    TRAIN_END + 1,
    SUPPORT_END,
    "Supporting User Data",
)

historical_user_records = pd.concat(
    [
        development_users,
        supporting_users,
    ],
    ignore_index=True,
)


print("\n" + "=" * 60)
print("LOADING HISTORICAL TRADES")
print("=" * 60)

# Reuse the existing trade loader from the previous section.
development_trades = load_trade_directory(
    DEVELOPMENT_DIR / "User Trades",
    is_unseen=False,
)

supporting_trades = load_trade_directory(
    SUPPORTING_DIR / "User Trades",
    is_unseen=False,
)

validate_campaign_range(
    development_trades,
    TRAIN_START,
    TRAIN_END,
    "Development trades",
)

validate_campaign_range(
    supporting_trades,
    TRAIN_END + 1,
    SUPPORT_END,
    "Supporting trades",
)

# Preserve the original historical trade data.
historical_trades_raw = pd.concat(
    [
        development_trades,
        supporting_trades,
    ],
    ignore_index=True,
)

historical_trades_raw["account_id"] = normalise_account(
    historical_trades_raw["account_id"]
)

if historical_trades_raw["account_id"].isna().any():
    raise ValueError(
        "Historical trade records contain missing account IDs."
    )


print("\n" + "=" * 60)
print("HISTORICAL DATA SUMMARY")
print("=" * 60)

print(
    "User Data rows:",
    f"{len(historical_user_records):,}",
)

print(
    "Trade rows:",
    f"{len(historical_trades_raw):,}",
)

print(
    "Campaigns:",
    historical_trades_raw["campaign_id"].nunique(),
)

print("Only campaigns 33–66 have been loaded.")
print("No sealed files accessed.")

LOADING HISTORICAL USER DATA
Loading 20 User Data files from /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/data/development/User Data...
  [1/20] Campaign 33: 500 user records
  [2/20] Campaign 34: 281 user records
  [3/20] Campaign 35: 500 user records
  [4/20] Campaign 36: 401 user records
  [5/20] Campaign 37: 455 user records
  [6/20] Campaign 38: 500 user records
  [7/20] Campaign 39: 340 user records
  [8/20] Campaign 40: 381 user records
  [9/20] Campaign 41: 223 user records
  [10/20] Campaign 42: 500 user records
  [11/20] Campaign 43: 500 user records
  [12/20] Campaign 44: 500 user records
  [13/20] Campaign 45: 500 user records
  [14/20] Campaign 46: 500 user records
  [15/20] Campaign 47: 500 user records
  [16/20] Campaign 48: 500 user records
  [17/20] Campaign 49: 500 user records
  [18/20] Campaign 50: 500 user records
  [19/20] Campaign 51: 500 user records
  [20/20] Campaign 52: 500 user records
Total User Data rows loaded: 9,081
L

In [ ]:
def make_person_id(email, campaign_id, account_id):
    """Generate a consistent email ID or isolated fallback ID."""
    if pd.notna(email):
        value = f"email:{email}"

        return (
            "email_"
            + sha256(
                value.encode("utf-8")
            ).hexdigest()
        )

    # Never link missing identities using account_id alone.
    value = (
        f"missing_identity:"
        f"{int(campaign_id)}:"
        f"{account_id}"
    )

    return (
        "fallback_"
        + sha256(
            value.encode("utf-8")
        ).hexdigest()
    )


def audit_user_identities(users, trades):
    """Audit User Data and map every historical trading account."""
    users = users.copy()
    trades = trades.copy()

    join_columns = [
        "campaign_id",
        "account_id",
    ]

    print("=" * 60)
    print("HISTORICAL USER IDENTITY AUDIT")
    print("=" * 60)

    # ==================================================
    # Validate and normalise input keys
    # ==================================================

    for frame, label in [
        (users, "User Data"),
        (trades, "User Trades"),
    ]:

        required_columns = set(join_columns)

        if label == "User Data":
            required_columns.add("email")

        missing_columns = (
            required_columns - set(frame.columns)
        )

        if missing_columns:
            raise ValueError(
                f"{label}: missing columns "
                f"{sorted(missing_columns)}"
            )

        if frame["campaign_id"].isna().any():
            raise ValueError(
                f"{label}: missing campaign IDs."
            )

        frame["campaign_id"] = (
            frame["campaign_id"].astype(int)
        )

        frame["account_id"] = normalise_account(
            frame["account_id"]
        )

        if frame["account_id"].isna().any():
            raise ValueError(
                f"{label}: missing account IDs."
            )

    users["email"] = normalise_email(
        users["email"]
    )

    # Ensure sealed campaigns have not entered this audit.
    allowed_campaigns = set(
        range(
            TRAIN_START,
            SUPPORT_END + 1,
        )
    )

    if not set(users["campaign_id"].unique()).issubset(
        allowed_campaigns
    ):
        raise ValueError(
            "User Data contains campaigns outside 33–66."
        )

    if not set(trades["campaign_id"].unique()).issubset(
        allowed_campaigns
    ):
        raise ValueError(
            "Trade data contains campaigns outside 33–66."
        )

    # ==================================================
    # Audit duplicate and conflicting User Data
    # ==================================================

    original_user_rows = len(users)

    missing_email_rows = int(
        users["email"].isna().sum()
    )

    duplicate_rows = int(
        users.duplicated(
            subset=join_columns
        ).sum()
    )

    email_counts = (
        users.groupby(join_columns)["email"]
        .nunique(dropna=True)
    )

    conflicting_email_mappings = int(
        (email_counts > 1).sum()
    )

    email_presence_counts = (
        users.assign(
            email_missing=users["email"].isna()
        )
        .groupby(join_columns)["email_missing"]
        .nunique()
    )

    inconsistent_email_mappings = int(
        (email_presence_counts > 1).sum()
    )

    # ==================================================
    # Cross-campaign identity audit
    # ==================================================

    valid_users = users.dropna(
        subset=["email"]
    ).copy()

    # Same account ID associated with different emails.
    account_email_counts = (
        valid_users.groupby("account_id")["email"]
        .nunique()
    )

    reused_account_ids = int(
        (account_email_counts > 1).sum()
    )

    # Same email associated with multiple account IDs.
    email_account_counts = (
        valid_users.groupby("email")["account_id"]
        .nunique()
    )

    emails_multiple_accounts = int(
        (email_account_counts > 1).sum()
    )

    # Same email associated with multiple accounts
    # within one campaign.
    campaign_email_counts = (
        valid_users.groupby(
            ["campaign_id", "email"]
        )["account_id"]
        .nunique()
    )

    same_campaign_shared_emails = int(
        (campaign_email_counts > 1).sum()
    )

    print("\nUser Data checks:")
    print("Raw user records:", original_user_rows)
    print("Missing email rows:", missing_email_rows)
    print("Duplicate mapping rows:", duplicate_rows)

    # ==================================================
    # Reject ambiguous mappings
    # ==================================================

    if conflicting_email_mappings > 0:
        raise ValueError(
            f"{conflicting_email_mappings} campaign-account "
            "pairs map to conflicting emails."
        )

    if inconsistent_email_mappings > 0:
        raise ValueError(
            f"{inconsistent_email_mappings} campaign-account "
            "pairs contain both missing and present emails."
        )

    # The remaining duplicate mappings agree on email.
    users = users.drop_duplicates(
        subset=join_columns
    ).copy()

    # ==================================================
    # Identify all accounts that actually have trades
    # ==================================================

    trading_accounts = (
        trades[join_columns]
        .drop_duplicates()
        .copy()
    )

    if trading_accounts.duplicated(join_columns).any():
        raise AssertionError(
            "Trading account keys must be unique."
        )

    if users.duplicated(join_columns).any():
        raise AssertionError(
            "User Data mapping keys must be unique."
        )

    print(
        "\nUnique trading campaign-account pairs:",
        len(trading_accounts),
    )

    # ==================================================
    # Join trading accounts to User Data
    # ==================================================

    # Left join is intentional: it preserves accounts
    # that are completely absent from User Data.
    mapping = trading_accounts.merge(
        users[
            join_columns + ["email"]
        ],
        on=join_columns,
        how="left",
        indicator=True,
        validate="one_to_one",
    )

    # Classify each trading account.
    missing_user_mask = (
        mapping["_merge"] == "left_only"
    )

    missing_email_mask = (
        (mapping["_merge"] == "both")
        & mapping["email"].isna()
    )

    email_based_mask = (
        (mapping["_merge"] == "both")
        & mapping["email"].notna()
    )

    mapping["identity_status"] = (
        "missing_user_record_fallback"
    )

    mapping.loc[
        missing_email_mask,
        "identity_status",
    ] = "missing_email_fallback"

    mapping.loc[
        email_based_mask,
        "identity_status",
    ] = "email_based"

    # Generate person IDs using one consistent function.
    mapping["person_id"] = [
        make_person_id(
            row.email,
            row.campaign_id,
            row.account_id,
        )
        for row in mapping.itertuples(index=False)
    ]

    # ==================================================
    # Missing-email accounts without trades
    # ==================================================

    # The join above includes only trading accounts.
    # This separate check preserves the audit count for
    # unused User Data records.
    user_usage = users[
        join_columns + ["email"]
    ].merge(
        trading_accounts.assign(
            has_trades=True
        ),
        on=join_columns,
        how="left",
        validate="one_to_one",
    )

    user_usage["has_trades"] = (
        user_usage["has_trades"]
        .fillna(False)
        .astype(bool)
    )

    missing_email_users = user_usage.loc[
        user_usage["email"].isna()
    ].copy()

    missing_email_summary = (
        missing_email_users.groupby("campaign_id")
        .agg(
            missing_email_accounts=("account_id", "size"),
            accounts_with_trades=("has_trades", "sum"),
        )
        .reset_index()
        .sort_values("campaign_id")
    )

    missing_email_summary[
        "accounts_without_trades"
    ] = (
        missing_email_summary["missing_email_accounts"]
        - missing_email_summary["accounts_with_trades"]
    )

    # ==================================================
    # Count trade rows affected by missing emails
    # ==================================================

    missing_email_keys = mapping.loc[
        missing_email_mask,
        join_columns,
    ].copy()

    missing_email_trades = trades.merge(
        missing_email_keys,
        on=join_columns,
        how="inner",
        validate="many_to_one",
    )

    # ==================================================
    # Count trade rows affected by missing User Data
    # ==================================================

    missing_user_keys = mapping.loc[
        missing_user_mask,
        join_columns,
    ].copy()

    missing_user_trades = trades.merge(
        missing_user_keys,
        on=join_columns,
        how="inner",
        validate="many_to_one",
    )

    missing_user_summary = (
        missing_user_trades.groupby("campaign_id")
        .agg(
            affected_trade_rows=("account_id", "size"),
            affected_accounts=("account_id", "nunique"),
        )
        .reset_index()
        .sort_values("campaign_id")
    )

    # ==================================================
    # Construct final audit summary
    # ==================================================

    identity_audit = pd.DataFrame([
        {
            "user_data_rows": original_user_rows,

            "unique_trading_accounts": len(
                trading_accounts
            ),

            "duplicate_user_data_rows": duplicate_rows,

            "account_ids_reused_by_different_emails":
                reused_account_ids,

            "emails_using_multiple_account_ids":
                emails_multiple_accounts,

            "same_email_multiple_accounts_in_campaign":
                same_campaign_shared_emails,

            "missing_email_user_data_rows":
                missing_email_rows,

            "missing_email_accounts_with_trades":
                int(missing_email_mask.sum()),

            "missing_email_accounts_without_trades":
                int(
                    (
                        ~missing_email_users["has_trades"]
                    ).sum()
                ),

            "missing_email_affected_trade_rows":
                len(missing_email_trades),

            "missing_user_record_accounts":
                int(missing_user_mask.sum()),

            "missing_user_record_affected_trade_rows":
                len(missing_user_trades),

            "email_based_account_mappings":
                int(email_based_mask.sum()),

            "unique_person_ids":
                mapping["person_id"].nunique(),
        }
    ])

    # ==================================================
    # Validate final identity mapping
    # ==================================================

    if len(mapping) != len(trading_accounts):
        raise AssertionError(
            "Identity mapping does not cover all trading accounts."
        )

    if mapping["person_id"].isna().any():
        raise AssertionError(
            "Some trading accounts have missing person IDs."
        )

    if mapping.duplicated(join_columns).any():
        raise AssertionError(
            "Duplicate campaign-account mappings found."
        )

    historical_users = mapping[
        join_columns
        + [
            "person_id",
            "identity_status",
        ]
    ].copy()

    # ==================================================
    # Display results
    # ==================================================

    print("\n" + "=" * 60)
    print("FINAL IDENTITY AUDIT")
    print("=" * 60)

    print("\nCombined audit:")
    display(identity_audit)

    print("\nMissing-email accounts by campaign:")
    display(missing_email_summary)

    print("\nMissing User Data accounts by campaign:")
    display(missing_user_summary)

    print("\nIdentity status counts:")

    identity_status_summary = (
        historical_users["identity_status"]
        .value_counts()
        .rename_axis("identity_status")
        .reset_index(name="account_mappings")
    )

    display(identity_status_summary)

    print(
        "\nIdentity mapping completed:",
        f"{len(historical_users):,} trading accounts."
    )

    return (
        historical_users,
        identity_audit,
        missing_email_summary,
        missing_user_summary,
    )


# ======================================================
# Run the historical identity audit
# ======================================================

(
    historical_users,
    identity_audit,
    missing_email_summary,
    missing_user_summary,
) = audit_user_identities(
    historical_user_records,
    historical_trades_raw,
)

print("\nHistorical identity audit completed.")
print("No sealed files accessed.")

HISTORICAL USER IDENTITY AUDIT

User Data checks:
Raw user records: 15878
Missing email rows: 391
Duplicate mapping rows: 3

Unique trading campaign-account pairs: 8165

FINAL IDENTITY AUDIT

Combined audit:


,user_data_rows,unique_trading_accounts,duplicate_user_data_rows,account_ids_reused_by_different_emails,emails_using_multiple_account_ids,same_email_multiple_accounts_in_campaign,missing_email_user_data_rows,missing_email_accounts_with_trades,missing_email_accounts_without_trades,missing_email_affected_trade_rows,missing_user_record_accounts,missing_user_record_affected_trade_rows,email_based_account_mappings,unique_person_ids
0,15878,8165,3,500,2523,57,391,23,368,116,6,28,8136,3579



Missing-email accounts by campaign:


,campaign_id,missing_email_accounts,accounts_with_trades,accounts_without_trades
0,44,65,11,54
1,50,56,3,53
2,51,136,0,136
3,56,77,9,68
4,59,57,0,57



Missing User Data accounts by campaign:


,campaign_id,affected_trade_rows,affected_accounts
0,33,10,3
1,34,1,1
2,37,17,2



Identity status counts:


,identity_status,account_mappings
0,email_based,8136
1,missing_email_fallback,23
2,missing_user_record_fallback,6



Identity mapping completed: 8,165 trading accounts.

Historical identity audit completed.
No sealed files accessed.


In [ ]:
def attach_person_ids(trades, identity_table):
    """Attach identities within campaigns without dropping trade rows."""
    join_columns = [
        "campaign_id",
        "account_id",
    ]

    trades = trades.copy()

    # Avoid attaching identities twice.
    if (
        "person_id" in trades.columns
        or "identity_status" in trades.columns
    ):
        raise ValueError(
            "Trade table already contains identity columns. "
            "Use historical_trades_raw instead."
        )

    # Normalise join keys.
    trades["campaign_id"] = (
        trades["campaign_id"].astype(int)
    )

    trades["account_id"] = normalise_account(
        trades["account_id"]
    )

    if trades["account_id"].isna().any():
        raise ValueError(
            "Trade rows contain missing account IDs."
        )

    # Confirm that each campaign-account has one identity.
    if identity_table.duplicated(join_columns).any():
        raise ValueError(
            "Identity table has duplicate campaign-account keys."
        )

    # Join strictly within campaigns.
    merged = trades.merge(
        identity_table[
            join_columns
            + [
                "person_id",
                "identity_status",
            ]
        ],
        on=join_columns,
        how="left",
        validate="many_to_one",
        sort=False,
    )

    # Ensure the join preserved every trade.
    if len(merged) != len(trades):
        raise AssertionError(
            "Identity join changed the trade row count."
        )

    # Every trading account should now have an identity.
    if merged["person_id"].isna().any():
        unmapped = (
            merged.loc[
                merged["person_id"].isna(),
                join_columns,
            ]
            .drop_duplicates()
        )

        raise ValueError(
            f"{len(unmapped)} campaign-account pairs "
            f"remain unmapped:\n"
            f"{unmapped.head(10).to_string(index=False)}"
        )

    print(
        "Person IDs attached successfully:",
        f"{len(merged):,} trades."
    )

    return merged


# Always join using the original trade data.
historical_trades = attach_person_ids(
    historical_trades_raw,
    historical_users,
)

# ======================================================
# Validate the final trade table
# ======================================================

assert len(historical_trades) == len(
    historical_trades_raw
)

assert historical_trades["person_id"].notna().all()

identity_trade_summary = (
    historical_trades.groupby("identity_status")
    .agg(
        trade_rows=("account_id", "size"),
        unique_person_ids=("person_id", "nunique"),
    )
    .reset_index()
)

print("\nTrade-level identity summary:")
display(identity_trade_summary)

print(
    "\nOriginal trade rows:",
    len(historical_trades_raw),
)

print(
    "Trade rows after identity join:",
    len(historical_trades),
)

print(
    "Missing person IDs:",
    historical_trades["person_id"].isna().sum(),
)

print("\nHistorical identity preparation completed.")
print("Sealed evaluation remains locked.")

Person IDs attached successfully: 46,520 trades.

Trade-level identity summary:


,identity_status,trade_rows,unique_person_ids
0,email_based,46376,3550
1,missing_email_fallback,116,23
2,missing_user_record_fallback,28,6



Original trade rows: 46520
Trade rows after identity join: 46520
Missing person IDs: 0

Historical identity preparation completed.
Sealed evaluation remains locked.


## Trade and idea reconstruction

In [ ]:
def attach_previous_completed_trade(
    trades,
):
    """Attach the most recent completed trade known at entry time."""

    result_frames = []

    grouping_columns = [
        "campaign_id",
        "account_id",
    ]

    for _, group in (
        trades.groupby(
            grouping_columns,
            dropna=False,
            sort=False,
        )
    ):
        current = (
            group
            .sort_values(
                [
                    "open_date_time",
                    "close_date_time",
                    "trade_row_id",
                ]
            )
            .copy()
        )

        completed = (
            group.loc[
                group["close_date_time"].notna()
                & group["net_profit"].notna()
            ]
            .sort_values(
                "close_date_time"
            )
            .copy()
        )

        previous_columns = [
            "close_date_time",
            "net_profit",
            "amount",
        ]

        if (
            "position_id"
            in completed.columns
        ):
            previous_columns.append(
                "position_id"
            )

        previous = (
            completed[
                previous_columns
            ]
            .rename(
                columns={
                    "close_date_time":
                        (
                            "previous_completed_"
                            "close_date_time"
                        ),
                    "net_profit":
                        (
                            "previous_completed_"
                            "net_profit"
                        ),
                    "amount":
                        (
                            "previous_completed_"
                            "amount"
                        ),
                    "position_id":
                        (
                            "previous_completed_"
                            "position_id"
                        ),
                }
            )
        )

        merged = pd.merge_asof(
            current.sort_values(
                "open_date_time"
            ),
            previous.sort_values(
                (
                    "previous_completed_"
                    "close_date_time"
                )
            ),
            left_on=(
                "open_date_time"
            ),
            right_on=(
                "previous_completed_"
                "close_date_time"
            ),
            direction="backward",
            allow_exact_matches=True,
        )

        result_frames.append(
            merged
        )

    result = pd.concat(
        result_frames,
        ignore_index=True,
    )

    result[
        "previous_completed_was_loss"
    ] = (
        result[
            "previous_completed_net_profit"
        ]
        < 0
    )

    result[
        "previous_completed_was_win"
    ] = (
        result[
            "previous_completed_net_profit"
        ]
        > 0
    )

    result[
        "reentry_gap_minutes"
    ] = (
        (
            result[
                "open_date_time"
            ]
            - result[
                (
                    "previous_completed_"
                    "close_date_time"
                )
            ]
        )
        .dt.total_seconds()
        / 60
    )

    return (
        result
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )


In [ ]:
def assign_trade_ideas(
    trades,
    maximum_gap_minutes=3.0,
):
    """Assign the Stage 1 directional trade-idea identifier."""

    result = (
        trades.copy()
    )

    group_columns = [
        "account_id",
        "campaign_id",
        "side",
    ]

    result = (
        result
        .sort_values(
            group_columns
            + [
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .copy()
    )

    idea_numbers = pd.Series(
        index=result.index,
        dtype="Int64",
    )

    for _, group in (
        result.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        group = group.sort_values(
            [
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )

        current_idea_number = 1
        current_idea_latest_close = (
            pd.NaT
        )

        for row in (
            group.itertuples()
        ):
            if pd.isna(
                current_idea_latest_close
            ):
                current_idea_latest_close = (
                    row.close_date_time
                )

            else:
                gap_minutes = (
                    (
                        row.open_date_time
                        - current_idea_latest_close
                    )
                    .total_seconds()
                    / 60
                )

                if (
                    gap_minutes
                    > maximum_gap_minutes
                ):
                    current_idea_number += 1

                    current_idea_latest_close = (
                        row.close_date_time
                    )

                else:
                    current_idea_latest_close = max(
                        current_idea_latest_close,
                        row.close_date_time,
                    )

            idea_numbers.loc[
                row.Index
            ] = (
                current_idea_number
            )

    result[
        "idea_number"
    ] = (
        idea_numbers
    )

    result[
        "idea_id"
    ] = (
        result[
            "account_id"
        ].astype(str)
        + "_"
        + result[
            "campaign_id"
        ].astype(str)
        + "_"
        + result[
            "side"
        ].astype(str)
        + "_"
        + result[
            "idea_number"
        ].astype(str)
    )

    return (
        result
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )


In [ ]:
def build_stage1_tables_from_trades(trades):
    """Reconstruct trade and idea masters from campaign-joined trade records."""
    trades = trades.copy()

    trades = (
        trades
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "source_row_number",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    trades[
        "trade_row_id"
    ] = np.arange(
        1,
        len(trades) + 1,
    )

    trades_with_previous_completed = (
        attach_previous_completed_trade(
            trades
        )
    )

    trades_with_ideas = (
        assign_trade_ideas(
            trades=trades,
            maximum_gap_minutes=3.0,
        )
    )

    trades_with_ideas = (
        trades_with_ideas
        .sort_values(
            [
                "idea_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .copy()
    )

    trades_with_ideas[
        "trade_number_within_idea"
    ] = (
        trades_with_ideas
        .groupby(
            "idea_id"
        )
        .cumcount()
        + 1
    )

    idea_features = (
        trades_with_ideas
        .groupby(
            "idea_id",
            as_index=False,
        )
        .agg(
            account_id=(
                "account_id",
                "first",
            ),
            person_id=("person_id", "first"),
            outcome_observed=("net_profit", lambda s: s.notna().all()),
            campaign_id=(
                "campaign_id",
                "first",
            ),
            side=(
                "side",
                "first",
            ),
            idea_start_time=(
                "open_date_time",
                "min",
            ),
            idea_end_time=(
                "close_date_time",
                "max",
            ),
            total_amount=(
                "amount",
                "sum",
            ),
            total_net_profit=(
                "net_profit",
                "sum",
            ),
        )
    )

    idea_features[
        "is_profitable_idea"
    ] = (
        idea_features[
            "total_net_profit"
        ]
        > 0
    )

    idea_features[
        "is_losing_idea"
    ] = (
        idea_features[
            "total_net_profit"
        ]
        < 0
    )

    idea_features[
        "is_breakeven_idea"
    ] = (
        idea_features[
            "total_net_profit"
        ]
        == 0
    )

    unseen_trade_row_ids = (
        trades.loc[
            trades[
                "_is_unseen"
            ],
            "trade_row_id",
        ]
        .tolist()
    )

    return {
        "trades":
            trades,
        "trades_with_previous_completed":
            (
                trades_with_previous_completed
            ),
        "trades_with_ideas":
            trades_with_ideas,
        "idea_features":
            idea_features,
        "unseen_trade_row_ids":
            unseen_trade_row_ids,
    }


# Feature engineering

## Pre-trade behavioural features

In [ ]:
def attach_past_event_count(
    target_table,
    event_table,
    group_columns,
    target_time_column,
    event_time_column,
    output_column,
):
    """Attach the number of strictly earlier events."""

    event_counts = (
        event_table
        .groupby(
            group_columns
            + [
                event_time_column
            ],
            dropna=False,
        )
        .size()
        .rename(
            "_events_at_timestamp"
        )
        .reset_index()
    )

    event_counts = (
        event_counts
        .sort_values(
            group_columns
            + [
                event_time_column
            ]
        )
    )

    event_counts[
        output_column
    ] = (
        event_counts
        .groupby(
            group_columns,
            dropna=False,
        )[
            "_events_at_timestamp"
        ]
        .cumsum()
    )

    output_groups = []

    for group_key, target_group in (
        target_table.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=(
                event_counts.index
            ),
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(
                value
            ):
                event_mask &= (
                    event_counts[
                        column
                    ].isna()
                )

            else:
                event_mask &= (
                    event_counts[
                        column
                    ].eq(
                        value
                    )
                )

        group_events = (
            event_counts.loc[
                event_mask,
                [
                    event_time_column,
                    output_column,
                ],
            ]
            .sort_values(
                event_time_column
            )
            .copy()
        )

        current_group = (
            target_group
            .sort_values(
                target_time_column
            )
            .copy()
        )

        if (
            group_events.empty
        ):
            current_group[
                output_column
            ] = 0

            output_groups.append(
                current_group
            )

            continue

        merge_time_column = (
            f"_past_{output_column}_time"
        )

        group_events = (
            group_events.rename(
                columns={
                    event_time_column:
                        merge_time_column
                }
            )
        )

        current_group = (
            pd.merge_asof(
                left=(
                    current_group
                ),
                right=(
                    group_events
                ),
                left_on=(
                    target_time_column
                ),
                right_on=(
                    merge_time_column
                ),
                direction="backward",
                allow_exact_matches=False,
            )
        )

        current_group[
            output_column
        ] = (
            current_group[
                output_column
            ]
            .fillna(0)
            .astype(int)
        )

        current_group = (
            current_group.drop(
                columns=[
                    merge_time_column
                ],
                errors="ignore",
            )
        )

        output_groups.append(
            current_group
        )

    return pd.concat(
        output_groups,
        ignore_index=True,
    )


In [ ]:
def attach_rolling_event_counts(
    target_table,
    event_table,
    group_columns,
    target_time_column,
    event_time_column,
    window_minutes,
    feature_prefix,
):
    """Attach counts of strictly earlier events in rolling windows."""

    result_groups = []

    for group_key, target_group in (
        target_table.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=(
                event_table.index
            ),
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(
                value
            ):
                event_mask &= (
                    event_table[
                        column
                    ].isna()
                )

            else:
                event_mask &= (
                    event_table[
                        column
                    ].eq(
                        value
                    )
                )

        group_events = (
            event_table.loc[
                event_mask,
                event_time_column,
            ]
            .dropna()
            .sort_values()
        )

        current_group = (
            target_group
            .sort_values(
                target_time_column
            )
            .copy()
        )

        target_times = (
            current_group[
                target_time_column
            ]
            .to_numpy(
                dtype="datetime64[ns]"
            )
        )

        event_times = (
            group_events
            .to_numpy(
                dtype="datetime64[ns]"
            )
        )

        for window in (
            window_minutes
        ):
            left_boundaries = (
                target_times
                - np.timedelta64(
                    window,
                    "m",
                )
            )

            left_indices = (
                np.searchsorted(
                    event_times,
                    left_boundaries,
                    side="left",
                )
            )

            right_indices = (
                np.searchsorted(
                    event_times,
                    target_times,
                    side="left",
                )
            )

            feature_column = (
                f"{feature_prefix}"
                f"_past_{window}_minutes"
            )

            current_group[
                feature_column
            ] = (
                right_indices
                - left_indices
            )

        result_groups.append(
            current_group
        )

    return pd.concat(
        result_groups,
        ignore_index=True,
    )


In [ ]:
def attach_entry_spacing_features(
    target_table,
    event_table,
    group_columns,
    target_time_column,
    event_time_column,
    tie_breaker_columns,
    previous_gap_column,
    past_median_gap_column,
    gap_ratio_column,
):
    """Attach pre-entry gap and historical median pace features."""

    output_groups = []

    for group_key, target_group in (
        target_table.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=(
                event_table.index
            ),
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(
                value
            ):
                event_mask &= (
                    event_table[
                        column
                    ].isna()
                )
            else:
                event_mask &= (
                    event_table[
                        column
                    ].eq(
                        value
                    )
                )

        group_events = (
            event_table.loc[
                event_mask
            ]
            .sort_values(
                [
                    event_time_column
                ]
                + tie_breaker_columns
            )
            .copy()
        )

        current_group = (
            target_group
            .sort_values(
                target_time_column
            )
            .copy()
        )

        if (
            group_events.empty
        ):
            current_group[
                previous_gap_column
            ] = np.nan

            current_group[
                past_median_gap_column
            ] = np.nan

            current_group[
                gap_ratio_column
            ] = np.nan

            output_groups.append(
                current_group
            )

            continue

        event_gap_column = (
            "_entry_gap_minutes"
        )

        group_events[
            event_gap_column
        ] = (
            group_events[
                event_time_column
            ]
            .diff()
            .dt.total_seconds()
            / 60
        )

        group_events[
            past_median_gap_column
        ] = (
            group_events[
                event_gap_column
            ]
            .expanding(
                min_periods=1
            )
            .median()
        )

        merge_time_column = (
            "_previous_event_time"
        )

        events_for_merge = (
            group_events[
                [
                    event_time_column,
                    past_median_gap_column,
                ]
            ]
            .rename(
                columns={
                    event_time_column:
                        merge_time_column
                }
            )
        )

        current_group = pd.merge_asof(
            left=(
                current_group
            ),
            right=(
                events_for_merge
            ),
            left_on=(
                target_time_column
            ),
            right_on=(
                merge_time_column
            ),
            direction="backward",
            allow_exact_matches=False,
        )

        current_group[
            previous_gap_column
        ] = (
            (
                current_group[
                    target_time_column
                ]
                - current_group[
                    merge_time_column
                ]
            )
            .dt.total_seconds()
            / 60
        )

        valid_ratio = (
            current_group[
                previous_gap_column
            ].ge(0)
            & current_group[
                past_median_gap_column
            ].gt(0)
        )

        current_group[
            gap_ratio_column
        ] = np.where(
            valid_ratio,
            (
                current_group[
                    previous_gap_column
                ]
                / current_group[
                    past_median_gap_column
                ]
            ),
            np.nan,
        )

        current_group = (
            current_group.drop(
                columns=[
                    merge_time_column
                ],
                errors="ignore",
            )
        )

        output_groups.append(
            current_group
        )

    return pd.concat(
        output_groups,
        ignore_index=True,
    )


In [ ]:
def attach_realized_challenge_state(
    target_table,
    realized_events,
):
    """Attach cumulative realized P&L known before each trade."""

    result_frames = []

    group_columns = [
        "campaign_id",
        "account_id",
    ]

    for group_key, target_group in (
        target_table.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=(
                realized_events.index
            ),
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(
                value
            ):
                event_mask &= (
                    realized_events[
                        column
                    ].isna()
                )

            else:
                event_mask &= (
                    realized_events[
                        column
                    ].eq(
                        value
                    )
                )

        group_events = (
            realized_events.loc[
                event_mask,
                [
                    "close_date_time",
                    "cumulative_realized_pnl",
                ],
            ]
            .sort_values(
                "close_date_time"
            )
            .copy()
        )

        current_group = (
            target_group
            .sort_values(
                "open_date_time"
            )
            .copy()
        )

        if (
            group_events.empty
        ):
            current_group[
                "realized_pnl_before_trade"
            ] = 0.0

            result_frames.append(
                current_group
            )

            continue

        group_events = (
            group_events.rename(
                columns={
                    "close_date_time":
                        (
                            "latest_realized_"
                            "close_before_trade"
                        )
                }
            )
        )

        current_group = pd.merge_asof(
            left=(
                current_group
            ),
            right=(
                group_events
            ),
            left_on=(
                "open_date_time"
            ),
            right_on=(
                "latest_realized_"
                "close_before_trade"
            ),
            direction="backward",
            allow_exact_matches=False,
        )

        current_group[
            "realized_pnl_before_trade"
        ] = (
            current_group[
                "cumulative_realized_pnl"
            ]
            .fillna(0.0)
        )

        current_group = (
            current_group.drop(
                columns=[
                    "cumulative_realized_pnl"
                ],
                errors="ignore",
            )
        )

        result_frames.append(
            current_group
        )

    return pd.concat(
        result_frames,
        ignore_index=True,
    )


In [ ]:
def attach_historical_position_size_features(
    target_table,
):
    """Attach historical median position-size features."""

    result_frames = []

    for _, group in (
        target_table.groupby(
            [
                "campaign_id",
                "account_id",
            ],
            sort=False,
            dropna=False,
        )
    ):
        current = (
            group
            .sort_values(
                [
                    "open_date_time",
                    "trade_row_id",
                ]
            )
            .copy()
        )

        historical_amounts = []
        same_timestamp_amounts = []
        previous_open_time = None

        medians = []

        for row in (
            current.itertuples()
        ):
            current_open_time = (
                row.open_date_time
            )

            if (
                previous_open_time
                is not None
                and current_open_time
                != previous_open_time
            ):
                historical_amounts.extend(
                    same_timestamp_amounts
                )

                same_timestamp_amounts = []

            medians.append(
                np.median(
                    historical_amounts
                )
                if historical_amounts
                else np.nan
            )

            same_timestamp_amounts.append(
                row.amount
            )

            previous_open_time = (
                current_open_time
            )

        current[
            "historical_median_amount_before_trade"
        ] = medians

        result_frames.append(
            current
        )

    result = pd.concat(
        result_frames,
        ignore_index=True,
    )

    result[
        "current_to_historical_median_amount_ratio"
    ] = (
        result[
            "amount"
        ]
        / result[
            "historical_median_amount_before_trade"
        ]
    )

    return result


In [ ]:
def attach_historical_sizing_consistency_features(
    target_table,
):
    """Attach historical position-size coefficient of variation."""

    result_frames = []

    for _, group in (
        target_table.groupby(
            [
                "campaign_id",
                "account_id",
            ],
            sort=False,
            dropna=False,
        )
    ):
        current = (
            group
            .sort_values(
                [
                    "open_date_time",
                    "trade_row_id",
                ]
            )
            .copy()
        )

        historical_amounts = []
        same_timestamp_amounts = []
        previous_open_time = None

        past_cvs = []

        for row in (
            current.itertuples()
        ):
            current_open_time = (
                row.open_date_time
            )

            if (
                previous_open_time
                is not None
                and current_open_time
                != previous_open_time
            ):
                historical_amounts.extend(
                    same_timestamp_amounts
                )

                same_timestamp_amounts = []

            past_count = len(
                historical_amounts
            )

            past_mean = (
                np.mean(
                    historical_amounts
                )
                if past_count >= 1
                else np.nan
            )

            past_std = (
                np.std(
                    historical_amounts,
                    ddof=0,
                )
                if past_count >= 2
                else np.nan
            )

            past_cv = (
                past_std / past_mean
                if (
                    past_count >= 2
                    and past_mean > 0
                )
                else np.nan
            )

            past_cvs.append(
                past_cv
            )

            same_timestamp_amounts.append(
                row.amount
            )

            previous_open_time = (
                current_open_time
            )

        current[
            "past_amount_cv"
        ] = (
            past_cvs
        )

        result_frames.append(
            current
        )

    return pd.concat(
        result_frames,
        ignore_index=True,
    )


## Person-keyed completed-idea history

In [ ]:
def build_historical_performance_features(
    idea_table,
):
    """Build historical completed-idea performance features."""

    all_ideas = idea_table.copy()
    idea_performance = (
        idea_table.loc[idea_table["outcome_observed"].eq(True)
                       & idea_table["idea_end_time"].notna()][
            [
                "idea_id",
                "person_id",
                "campaign_id",
                "idea_start_time",
                "idea_end_time",
                "total_amount",
                "total_net_profit",
                "is_profitable_idea",
                "is_losing_idea",
                "is_breakeven_idea",
            ]
        ]
        .copy()
    )

    idea_performance[
        "idea_profit_per_lot"
    ] = (
        idea_performance[
            "total_net_profit"
        ]
        / idea_performance[
            "total_amount"
        ]
    )

    idea_performance[
        "_win_count"
    ] = (
        idea_performance[
            "is_profitable_idea"
        ]
        .astype(int)
    )

    idea_performance[
        "_loss_count"
    ] = (
        idea_performance[
            "is_losing_idea"
        ]
        .astype(int)
    )

    idea_performance[
        "_idea_count"
    ] = 1

    idea_performance[
        "_win_profit"
    ] = (
        idea_performance[
            "idea_profit_per_lot"
        ]
        .where(
            idea_performance[
                "is_profitable_idea"
            ],
            0.0,
        )
    )

    idea_performance[
        "_loss_abs_profit"
    ] = (
        idea_performance[
            "idea_profit_per_lot"
        ]
        .abs()
        .where(
            idea_performance[
                "is_losing_idea"
            ],
            0.0,
        )
    )

    if idea_performance.empty:
        current_ideas = all_ideas[["idea_id", "person_id", "campaign_id", "idea_start_time"]].copy()
        for name in ["past_completed_idea_count", "past_idea_win_rate", "past_mean_win_profit_per_lot",
                     "past_mean_loss_abs_profit_per_lot", "past_payoff_ratio"]:
            current_ideas[name] = np.nan
        return current_ideas

    events = (
        idea_performance
        .groupby(
            [
                "person_id",
                "idea_end_time",
            ],
            as_index=False,
        )
        .agg(
            completed_idea_count=(
                "_idea_count",
                "sum",
            ),
            completed_win_count=(
                "_win_count",
                "sum",
            ),
            completed_loss_count=(
                "_loss_count",
                "sum",
            ),
            completed_win_profit_sum=(
                "_win_profit",
                "sum",
            ),
            completed_loss_abs_profit_sum=(
                "_loss_abs_profit",
                "sum",
            ),
        )
        .sort_values(
            [
                "person_id",
                "idea_end_time",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    cumulative_mapping = {
        "past_completed_idea_count":
            "completed_idea_count",
        "past_winning_idea_count":
            "completed_win_count",
        "past_losing_idea_count":
            "completed_loss_count",
        "past_win_profit_sum":
            "completed_win_profit_sum",
        "past_loss_abs_profit_sum":
            (
                "completed_loss_"
                "abs_profit_sum"
            ),
    }

    for (
        output_column,
        source_column,
    ) in (
        cumulative_mapping.items()
    ):
        events[
            output_column
        ] = (
            events
            .groupby(
                "person_id"
            )[
                source_column
            ]
            .cumsum()
        )

    current_ideas = (
        all_ideas[
            [
                "idea_id",
                "person_id",
                "campaign_id",
                "idea_start_time",
            ]
        ]
        .sort_values(
            [
                "idea_start_time",
                "person_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    events = (
        events
        .sort_values(
            [
                "idea_end_time",
                "person_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    historical = pd.merge_asof(
        current_ideas,
        events[
            [
                "person_id",
                "idea_end_time",
                "past_completed_idea_count",
                "past_winning_idea_count",
                "past_losing_idea_count",
                "past_win_profit_sum",
                "past_loss_abs_profit_sum",
            ]
        ],
        left_on=(
            "idea_start_time"
        ),
        right_on=(
            "idea_end_time"
        ),
        by="person_id",
        direction="backward",
        allow_exact_matches=False,
    )

    historical[
        "past_idea_win_rate"
    ] = (
        historical[
            "past_winning_idea_count"
        ]
        / historical[
            "past_completed_idea_count"
        ]
    )

    historical[
        "past_mean_win_profit_per_lot"
    ] = (
        historical[
            "past_win_profit_sum"
        ]
        / historical[
            "past_winning_idea_count"
        ]
    )

    historical[
        "past_mean_loss_abs_profit_per_lot"
    ] = (
        historical[
            "past_loss_abs_profit_sum"
        ]
        / historical[
            "past_losing_idea_count"
        ]
    )

    historical[
        "past_payoff_ratio"
    ] = (
        historical[
            "past_mean_win_profit_per_lot"
        ]
        / historical[
            "past_mean_loss_abs_profit_per_lot"
        ]
    )

    return historical


## Model feature tables

In [ ]:
def build_model_features(
    stage1_tables,
):
    """Recreate the 22 pre-trade features used by the frozen model."""

    trades = (
        stage1_tables[
            "trades"
        ]
        .copy()
    )

    trades_with_ideas = (
        stage1_tables[
            "trades_with_ideas"
        ]
        .copy()
    )

    previous_completed = (
        stage1_tables[
            "trades_with_previous_completed"
        ]
        .copy()
    )

    idea_features = (
        stage1_tables[
            "idea_features"
        ]
        .copy()
    )

    trade_features = (
        trades_with_ideas
        .copy()
    )

    # --------------------------------------------------------
    # Loss response
    # --------------------------------------------------------

    previous_columns = [
        "trade_row_id",
        "previous_completed_close_date_time",
        "previous_completed_net_profit",
        "previous_completed_amount",
        "previous_completed_was_loss",
        "previous_completed_was_win",
        "reentry_gap_minutes",
    ]

    trade_features = (
        trade_features.merge(
            previous_completed[
                previous_columns
            ],
            on="trade_row_id",
            how="left",
            validate="one_to_one",
        )
    )

    trade_features[
        "current_to_previous_amount_ratio"
    ] = (
        trade_features[
            "amount"
        ]
        / trade_features[
            "previous_completed_amount"
        ]
    )

    trade_features[
        "current_to_previous_amount_ratio"
    ] = (
        trade_features[
            "current_to_previous_amount_ratio"
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )

    trade_features[
        "post_loss_entry"
    ] = (
        trade_features[
            "previous_completed_was_loss"
        ]
        .fillna(False)
        .astype(bool)
    )

    trade_features[
        "post_loss_reentry_gap_minutes"
    ] = (
        trade_features[
            "reentry_gap_minutes"
        ]
        .where(
            trade_features[
                "post_loss_entry"
            ]
        )
    )

    trade_features[
        "post_loss_amount_ratio"
    ] = (
        trade_features[
            "current_to_previous_amount_ratio"
        ]
        .where(
            trade_features[
                "post_loss_entry"
            ]
        )
    )

    # --------------------------------------------------------
    # Activity
    # --------------------------------------------------------

    trade_features = (
        attach_past_event_count(
            target_table=(
                trade_features
            ),
            event_table=trades,
            group_columns=[
                "account_id",
                "campaign_id",
            ],
            target_time_column=(
                "open_date_time"
            ),
            event_time_column=(
                "open_date_time"
            ),
            output_column=(
                "past_trade_open_count"
            ),
        )
    )

    trade_features[
        "first_trade_open_date_time"
    ] = (
        trade_features
        .groupby(
            [
                "account_id",
                "campaign_id",
            ]
        )[
            "open_date_time"
        ]
        .transform(
            "min"
        )
    )

    trade_features[
        "elapsed_active_hours"
    ] = (
        (
            trade_features[
                "open_date_time"
            ]
            - trade_features[
                "first_trade_open_date_time"
            ]
        )
        .dt.total_seconds()
        / 3600
    )

    valid_elapsed = (
        trade_features[
            "elapsed_active_hours"
        ]
        > 0
    )

    trade_features[
        "past_trades_opened_per_active_hour"
    ] = np.where(
        valid_elapsed,
        (
            trade_features[
                "past_trade_open_count"
            ]
            / trade_features[
                "elapsed_active_hours"
            ]
        ),
        np.nan,
    )

    trade_features = (
        attach_rolling_event_counts(
            target_table=(
                trade_features
            ),
            event_table=trades,
            group_columns=[
                "account_id",
                "campaign_id",
            ],
            target_time_column=(
                "open_date_time"
            ),
            event_time_column=(
                "open_date_time"
            ),
            window_minutes=[
                30,
            ],
            feature_prefix=(
                "trades_opened"
            ),
        )
    )

    # --------------------------------------------------------
    # Trade timing
    # --------------------------------------------------------

    trade_features = (
        attach_entry_spacing_features(
            target_table=(
                trade_features
            ),
            event_table=trades,
            group_columns=[
                "account_id",
                "campaign_id",
            ],
            target_time_column=(
                "open_date_time"
            ),
            event_time_column=(
                "open_date_time"
            ),
            tie_breaker_columns=[
                "trade_row_id",
            ],
            previous_gap_column=(
                "minutes_since_previous_trade_open"
            ),
            past_median_gap_column=(
                "past_median_trade_open_gap_minutes"
            ),
            gap_ratio_column=(
                "trade_open_gap_to_past_median_ratio"
            ),
        )
    )

    idea_start_events = (
        idea_features[
            [
                "account_id",
                "campaign_id",
                "idea_id",
                "idea_start_time",
            ]
        ]
        .drop_duplicates(
            subset=[
                "idea_id"
            ]
        )
        .copy()
    )

    idea_features = (
        attach_entry_spacing_features(
            target_table=(
                idea_features
            ),
            event_table=(
                idea_start_events
            ),
            group_columns=[
                "account_id",
                "campaign_id",
            ],
            target_time_column=(
                "idea_start_time"
            ),
            event_time_column=(
                "idea_start_time"
            ),
            tie_breaker_columns=[
                "idea_id",
            ],
            previous_gap_column=(
                "minutes_since_previous_idea_start"
            ),
            past_median_gap_column=(
                "past_median_idea_start_gap_minutes"
            ),
            gap_ratio_column=(
                "idea_start_gap_to_past_median_ratio"
            ),
        )
    )

    trade_features = (
        trade_features.merge(
            idea_features[
                [
                    "idea_id",
                    "minutes_since_previous_idea_start",
                    "past_median_idea_start_gap_minutes",
                    "idea_start_gap_to_past_median_ratio",
                ]
            ],
            on="idea_id",
            how="left",
            validate="many_to_one",
        )
    )

    # --------------------------------------------------------
    # Challenge state
    # --------------------------------------------------------

    realized_events = (
        trades[
            [
                "account_id",
                "campaign_id",
                "close_date_time",
                "net_profit",
            ]
        ]
        .dropna(
            subset=[
                "close_date_time",
                "net_profit",
            ]
        )
        .groupby(
            [
                "account_id",
                "campaign_id",
                "close_date_time",
            ],
            as_index=False,
        )[
            "net_profit"
        ]
        .sum()
        .rename(
            columns={
                "net_profit":
                    "realized_pnl_at_close"
            }
        )
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "close_date_time",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    realized_events[
        "cumulative_realized_pnl"
    ] = (
        realized_events
        .groupby(
            [
                "campaign_id",
                "account_id",
            ]
        )[
            "realized_pnl_at_close"
        ]
        .cumsum()
    )

    trade_features = (
        attach_realized_challenge_state(
            target_table=(
                trade_features
            ),
            realized_events=(
                realized_events
            ),
        )
    )

    trade_features[
        "realized_distance_to_drawdown_limit"
    ] = (
        trade_features[
            "realized_pnl_before_trade"
        ]
        - REALIZED_DRAWDOWN_BOUNDARY
    )

    # --------------------------------------------------------
    # Position sizing
    # --------------------------------------------------------

    trade_features = (
        attach_historical_position_size_features(
            trade_features
        )
    )

    trade_features = (
        attach_historical_sizing_consistency_features(
            trade_features
        )
    )

    # --------------------------------------------------------
    # Historical idea performance
    # --------------------------------------------------------

    historical_idea_features = (
        build_historical_performance_features(
            idea_features
        )
    )

    historical_columns = [
        "idea_id",
        "past_completed_idea_count",
        "past_idea_win_rate",
        "past_mean_win_profit_per_lot",
        "past_mean_loss_abs_profit_per_lot",
        "past_payoff_ratio",
    ]

    trade_features = (
        trade_features.merge(
            historical_idea_features[
                historical_columns
            ],
            on="idea_id",
            how="left",
            validate="many_to_one",
        )
    )

    # Replace invalid numerical values exactly as missing values.
    trade_features = (
        trade_features.replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )

    if (
        "reverse_profit"
        in trade_features.columns
        and "amount"
        in trade_features.columns
    ):
        trade_features[
            "reverse_profit_per_lot"
        ] = (
            trade_features[
                "reverse_profit"
            ]
            / trade_features[
                "amount"
            ]
        )

        trade_features[
            "reverse_profit_per_lot"
        ] = (
            trade_features[
                "reverse_profit_per_lot"
            ]
            .replace(
                [
                    np.inf,
                    -np.inf,
                ],
                np.nan,
            )
        )

    return (
        trade_features
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )


# Historical preparation

In [ ]:
# ======================================================
# Prepare historical model features: Campaigns 33–66
# ======================================================

print("Preparing historical features for campaigns 33–66...")

# Reuse the historical trades prepared in the identity-audit section.
# Do not reload the files or attach person IDs a second time.
historical_trades = historical_trades.copy()

# ------------------------------------------------------
# 1. Validate campaigns and identities
# ------------------------------------------------------

expected_campaigns = set(
    range(TRAIN_START, SUPPORT_END + 1)
)

actual_campaigns = set(
    historical_trades["campaign_id"].unique()
)

if actual_campaigns != expected_campaigns:
    raise ValueError(
        "Historical campaigns do not match 33–66. "
        f"Missing: {sorted(expected_campaigns - actual_campaigns)}. "
        f"Unexpected: {sorted(actual_campaigns - expected_campaigns)}."
    )

if historical_trades["person_id"].isna().any():
    raise ValueError(
        "Historical trades contain missing person IDs."
    )

# The reconstruction function requires this flag.
# All campaigns 33–66 are historical, not sealed.
historical_trades["_is_unseen"] = False

# ------------------------------------------------------
# 2. Reconstruct trades and ideas
# ------------------------------------------------------

stage1_historical = build_stage1_tables_from_trades(
    historical_trades
)

# ------------------------------------------------------
# 3. Build model features
# ------------------------------------------------------

historical_features = build_model_features(
    stage1_historical
)

# ------------------------------------------------------
# 4. Validate feature table
# ------------------------------------------------------

if len(historical_features) != len(historical_trades):
    raise AssertionError(
        "Feature engineering changed the historical trade row count."
    )

missing_feature_columns = (
    set(FEATURE_COLUMNS)
    - set(historical_features.columns)
)

if missing_feature_columns:
    raise AssertionError(
        f"Missing frozen features: {missing_feature_columns}"
    )

if historical_features["reverse_profit_per_lot"].isna().any():
    raise ValueError(
        "Missing training outcomes in historical campaigns 33–66."
    )

print(
    f"Prepared {len(historical_features):,} historical feature rows "
    f"with {len(FEATURE_COLUMNS)} model features."
)

# ------------------------------------------------------
# 5. Prepare campaign deadlines
# ------------------------------------------------------

campaign_dates = historical_trades[
    ["campaign_id", "campaign_date"]
].drop_duplicates()

if (
    campaign_dates["campaign_date"].isna().any()
    or campaign_dates["campaign_id"].duplicated().any()
):
    raise ValueError(
        "Missing or conflicting campaign start dates."
    )

CAMPAIGN_DEADLINES_UTC = {
    row.campaign_id: (
        pd.Timestamp(row.campaign_date).normalize()
        .tz_localize(CHALLENGE_TIMEZONE)
        + pd.Timedelta(
            hours=CHALLENGE_START_HOUR + CHALLENGE_DURATION_HOURS
        )
    ).tz_convert("UTC")
    for row in campaign_dates.itertuples(index=False)
}

print(
    f"Prepared deadlines for "
    f"{len(CAMPAIGN_DEADLINES_UTC)} campaigns."
)

Preparing historical features for campaigns 33–66...
Prepared 46,520 historical feature rows with 22 model features.
Prepared deadlines for 34 campaigns.


# Frozen model specification

In [ ]:
# ======================================================
# Frozen model specification
# ======================================================

BANNED_MODEL_COLUMNS = {
    "profit",
    "net_profit",
    "reverse_profit",
    "reverse_profit_per_lot",
    "sl_price",
    "tp_price",
    "person_id",
    "account_id",
    "email",
    "ip_address",
}

assert (
    len(FEATURE_COLUMNS) == 22
    and not BANNED_MODEL_COLUMNS.intersection(FEATURE_COLUMNS)
)


def new_regressor():
    """Instantiate the fixed Stage 3 LightGBM specification."""
    return LGBMRegressor(**MODEL_PARAMETERS)


def fit_fixed_regressor(data):
    """Fit the registered model using historical campaign data."""
    if data.empty or data["reverse_profit_per_lot"].isna().any():
        raise ValueError(
            "Training requires complete historical labels."
        )

    model = new_regressor()

    model.fit(
        data[FEATURE_COLUMNS],
        data["reverse_profit_per_lot"],
    )

    return model


def fit_and_score_campaign(feature_data, campaign_id):
    """Refit on earlier campaigns and score the current campaign."""

    # Train only on campaigns preceding the campaign being predicted.
    history = feature_data.loc[
        feature_data["campaign_id"] < campaign_id
    ].copy()

    current = feature_data.loc[
        feature_data["campaign_id"] == campaign_id
    ].copy()

    if history.empty or current.empty:
        raise ValueError(
            f"No train/test data for campaign {campaign_id}."
        )

    # Historical outcomes are required for model fitting.
    if history["reverse_profit_per_lot"].isna().any():
        raise ValueError(
            f"Prior campaign labels are missing before "
            f"campaign {campaign_id}; cannot refit walk-forward."
        )

    # Fit the same model specification on the available history.
    model = fit_fixed_regressor(history)

    # Calibrate the threshold using training predictions only.
    training_scores = model.predict(
        history[FEATURE_COLUMNS]
    )

    threshold = float(
        np.quantile(training_scores, QUANTILE)
    )

    # Predict the current campaign without fitting on its outcomes.
    current["predicted_score"] = model.predict(
        current[FEATURE_COLUMNS]
    )

    current["decision_threshold"] = threshold

    current["raw_fade_decision"] = (
        current["predicted_score"] >= threshold
    )

    print(
        f"Campaign {campaign_id}: "
        f"trained={len(history):,}, "
        f"scored={len(current):,}, "
        f"threshold={threshold:.6f}, "
        f"coverage={current['raw_fade_decision'].mean():.2%}"
    )

    return current


print("Frozen model specification loaded.")

Frozen model specification loaded.


# Supporting walk-forward

In [ ]:
# Supporting walk-forward evaluation: Campaigns 53–66

print("Generating supporting walk-forward predictions...")

support_frames = []

for campaign_id in range(SUPPORT_START, SUPPORT_END + 1):
    campaign_predictions = fit_and_score_campaign(
        historical_features,
        campaign_id,
    )
    support_frames.append(campaign_predictions)

support_predictions = pd.concat(
    support_frames,
    ignore_index=True,
)

print(
    "Supporting walk-forward completed: "
    f"campaigns {SUPPORT_START}–{SUPPORT_END}."
)

Generating supporting walk-forward predictions...


Campaign 53: trained=25,165, scored=1,063, threshold=60.439365, coverage=9.97%
Campaign 54: trained=26,228, scored=1,511, threshold=57.532725, coverage=10.46%
Campaign 55: trained=27,739, scored=1,222, threshold=58.682902, coverage=7.36%
Campaign 56: trained=28,961, scored=1,732, threshold=51.320125, coverage=8.43%
Campaign 57: trained=30,693, scored=1,256, threshold=39.850729, coverage=8.52%
Campaign 58: trained=31,949, scored=1,263, threshold=38.202976, coverage=9.82%
Campaign 59: trained=33,212, scored=1,490, threshold=35.634304, coverage=8.46%
Campaign 60: trained=34,702, scored=1,575, threshold=40.757560, coverage=8.44%
Campaign 61: trained=36,277, scored=1,573, threshold=38.911996, coverage=10.49%
Campaign 62: trained=37,850, scored=1,602, threshold=35.700989, coverage=7.37%
Campaign 63: trained=39,452, scored=1,715, threshold=34.708084, coverage=11.49%
Campaign 64: trained=41,167, scored=1,696, threshold=33.029040, coverage=9.73%
Campaign 65: trained=42,863, scored=1,746, thresh

# Stand-down filter development

## Candidate definitions

In [ ]:
# ======================================================
# Stand-down filter: Historical position sizing
# ======================================================

def person_closed_lot_median(data, trades):
    """Find each person's median lot among strictly earlier closed trades."""

    completed = trades.loc[
        trades["close_date_time"].notna()
        & trades["amount"].notna(),
        ["person_id", "close_date_time", "amount"],
    ].copy()

    completed = completed.sort_values("close_date_time")

    output = pd.Series(np.nan, index=data.index, dtype=float)

    for person, entries in data.groupby("person_id", sort=False):
        events = completed.loc[
            completed["person_id"].eq(person)
        ]

        if events.empty:
            continue

        event_times = pd.DatetimeIndex(events["close_date_time"])
        lots = events["amount"].to_numpy(dtype=float)

        for index, opened in entries["open_date_time"].items():
            cutoff = event_times.searchsorted(opened, side="left")

            if cutoff:
                output.at[index] = float(np.median(lots[:cutoff]))

    return output

In [ ]:
# ======================================================
# Stand-down filter: Behavioural states
# ======================================================

def attach_filter_states(data, idea_features, trades):
    """Construct behavioural states using information available before entry."""

    data = data.copy()

    # --------------------------------------------------
    # 1. Size-up: Person-level history across campaigns
    # --------------------------------------------------

    data["person_past_closed_median_lot"] = person_closed_lot_median(
        data,
        trades,
    )

    data["size_up_ratio"] = (
        data["amount"] / data["person_past_closed_median_lot"]
    ).replace([np.inf, -np.inf], np.nan)

    # --------------------------------------------------
    # 2. Hot streak: Reset for every campaign
    # --------------------------------------------------

    ideas = idea_features.loc[
        idea_features["outcome_observed"].eq(True)
        & idea_features["idea_end_time"].notna(),
        [
            "campaign_id",
            "person_id",
            "idea_end_time",
            "total_net_profit",
            "idea_id",
        ],
    ].copy()

    ideas = ideas.sort_values(
        ["campaign_id", "person_id", "idea_end_time", "idea_id"]
    )

    events = []

    for (campaign_id, person_id), group in ideas.groupby(
        ["campaign_id", "person_id"],
        sort=False,
    ):
        streak = 0

        for close_time, simultaneous in group.groupby(
            "idea_end_time",
            sort=True,
        ):
            if simultaneous["total_net_profit"].gt(0).all():
                streak += len(simultaneous)
            else:
                streak = 0

            events.append({
                "campaign_id": campaign_id,
                "person_id": person_id,
                "streak_event_time": close_time,
                "hot_streak_completed_wins": streak,
            })

    if events:
        event_df = pd.DataFrame(events).sort_values(
            "streak_event_time"
        )

        data = pd.merge_asof(
            data.sort_values("open_date_time"),
            event_df,
            left_on="open_date_time",
            right_on="streak_event_time",
            by=["campaign_id", "person_id"],
            direction="backward",
            allow_exact_matches=False,
        )
    else:
        data["hot_streak_completed_wins"] = np.nan

    data["hot_streak_completed_wins"] = (
        data["hot_streak_completed_wins"].fillna(0)
    )

    # --------------------------------------------------
    # 3. Near target and desperate pace
    # --------------------------------------------------

    if PROFIT_TARGET_AMOUNT is not None:
        data["profit_still_needed"] = (
            PROFIT_TARGET_AMOUNT
            - data["realized_pnl_before_trade"]
        )

        data["near_target_remaining_profit"] = (
            data["profit_still_needed"]
        )

        deadline = pd.to_datetime(
            data["campaign_id"].map(CAMPAIGN_DEADLINES_UTC),
            utc=True,
            errors="coerce",
        )

        if deadline.isna().any():
            raise ValueError(
                "Missing campaign deadlines for desperate-pace calculation."
            )

        hours_remaining = (
            deadline - data["open_date_time"]
        ).dt.total_seconds() / 3600

        data["desperate_pace_profit_per_hour"] = (
            data["profit_still_needed"]
            / hours_remaining.where(hours_remaining > 0)
        )

    else:
        data["near_target_remaining_profit"] = np.nan
        data["desperate_pace_profit_per_hour"] = np.nan

    return data.drop(
        columns=["streak_event_time"],
        errors="ignore",
    )

In [ ]:
# ======================================================
# Stand-down filter: Candidate masks
# ======================================================

def filter_candidate_masks(data):
    """Evaluate each candidate using its configured cutoff."""

    masks = {}

    size_up_cutoff = FILTER_CUTOFFS.get("size_up_ratio")

    if size_up_cutoff is not None:
        masks["size_up"] = (
            data["size_up_ratio"]
            .ge(size_up_cutoff)
            .fillna(False)
        )

    hot_streak_cutoff = FILTER_CUTOFFS.get(
        "hot_streak_completed_wins"
    )

    if hot_streak_cutoff is not None:
        masks["hot_streak"] = (
            data["hot_streak_completed_wins"]
            .ge(hot_streak_cutoff)
            .fillna(False)
        )

    near_target_cutoff = FILTER_CUTOFFS.get(
        "near_target_remaining_profit"
    )

    if near_target_cutoff is not None:
        if data["near_target_remaining_profit"].notna().sum() == 0:
            raise ValueError(
                "Near-target candidate enabled without profit-target data."
            )

        masks["near_target"] = (
            data["near_target_remaining_profit"]
            .le(near_target_cutoff)
            .fillna(False)
        )

    desperate_cutoff = FILTER_CUTOFFS.get(
        "desperate_pace_profit_per_hour"
    )

    if desperate_cutoff is not None:
        if data["desperate_pace_profit_per_hour"].notna().sum() == 0:
            raise ValueError(
                "Desperate-pace candidate enabled without deadline data."
            )

        masks["desperate_pace"] = (
            data["desperate_pace_profit_per_hour"]
            .ge(desperate_cutoff)
            .fillna(False)
        )

    return masks

In [ ]:
# ======================================================
# Stand-down filter development: Cutoff grid selection
# ======================================================

# Proposed grid: confirm before evaluating performance.
FILTER_GRID = {
    "size_up": [1.5, 2.0, 3.0],
    "hot_streak": [2, 3, 4],
    "near_target": [100.0, 50.0, 25.0],
    "desperate_pace": [50.0, 100.0, 200.0],
}

MIN_FILTER_TRADES = 30
MIN_FILTER_PERSONS = 10
BOOTSTRAP_SAMPLES = 2000


def bootstrap_filter_ci(data, n_bootstrap=BOOTSTRAP_SAMPLES):
    """Calculate a 95% person-cluster CI for lot-weighted reverse P&L."""

    person_totals = data.groupby("person_id").agg(
        reverse_profit=("reverse_profit", "sum"),
        amount=("amount", "sum"),
    )

    values = person_totals[["reverse_profit", "amount"]].to_numpy()
    n_persons = len(values)

    rng = np.random.default_rng(SEED)
    estimates = np.empty(n_bootstrap)

    for i in range(n_bootstrap):
        sample = values[
            rng.integers(0, n_persons, size=n_persons)
        ]

        estimates[i] = (
            sample[:, 0].sum() / sample[:, 1].sum()
        )

    return np.quantile(estimates, [0.025, 0.975])


# Only evaluate trades that passed the supporting walk-forward gate.
gated_support = support_with_states.loc[
    support_with_states["raw_fade_decision"]
].copy()

candidate_definitions = {
    "size_up": ("size_up_ratio", "ge"),
    "hot_streak": ("hot_streak_completed_wins", "ge"),
    "near_target": ("near_target_remaining_profit", "le"),
    "desperate_pace": ("desperate_pace_profit_per_hour", "ge"),
}

results = []

for candidate, cutoffs in FILTER_GRID.items():
    column, direction = candidate_definitions[candidate]

    for cutoff in cutoffs:
        if direction == "ge":
            mask = gated_support[column].ge(cutoff)
        else:
            mask = gated_support[column].le(cutoff)

        excluded = gated_support.loc[mask].copy()

        n_trades = len(excluded)
        n_persons = excluded["person_id"].nunique()

        if (
            n_trades < MIN_FILTER_TRADES
            or n_persons < MIN_FILTER_PERSONS
        ):
            results.append({
                "candidate": candidate,
                "cutoff": cutoff,
                "trades": n_trades,
                "persons": n_persons,
                "rp_per_lot": np.nan,
                "ci_lower": np.nan,
                "ci_upper": np.nan,
                "qualifies": False,
            })
            continue

        rp_per_lot = (
            excluded["reverse_profit"].sum()
            / excluded["amount"].sum()
        )

        ci_lower, ci_upper = bootstrap_filter_ci(excluded)

        results.append({
            "candidate": candidate,
            "cutoff": cutoff,
            "trades": n_trades,
            "persons": n_persons,
            "rp_per_lot": rp_per_lot,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
            "qualifies": (
                rp_per_lot < 0
                and ci_upper < 0
            ),
        })


filter_grid_results = pd.DataFrame(results)

display(
    filter_grid_results.sort_values(
        ["candidate", "cutoff"]
    )
)

# Select the first qualifying cutoff in each predefined grid.
# Grid order goes from less restrictive to more restrictive.
selected_cutoffs = {}

for candidate, cutoffs in FILTER_GRID.items():
    candidate_results = filter_grid_results.loc[
        filter_grid_results["candidate"].eq(candidate)
        & filter_grid_results["qualifies"]
    ]

    selected_cutoffs[candidate] = (
        candidate_results.iloc[0]["cutoff"]
        if not candidate_results.empty
        else None
    )

print("Selected development cutoffs:", selected_cutoffs)

,candidate,cutoff,trades,persons,rp_per_lot,ci_lower,ci_upper,qualifies
9,desperate_pace,50.0,152,101,-38.180798,-119.034678,38.893586,False
10,desperate_pace,100.0,22,19,NaN,NaN,NaN,False
11,desperate_pace,200.0,15,13,NaN,NaN,NaN,False
3,hot_streak,2.0,346,213,-60.135985,-123.653904,7.821240,False
4,hot_streak,3.0,141,88,-42.661181,-125.656500,39.508124,False
5,hot_streak,4.0,56,40,-111.900892,-235.288286,11.590199,False
8,near_target,25.0,31,26,3.612371,-81.312987,108.026130,False
7,near_target,50.0,57,43,-19.362971,-84.138400,58.825592,False
6,near_target,100.0,121,75,4.558994,-86.267515,112.425136,False
0,size_up,1.5,633,413,-38.970155,-82.386513,1.869950,False


Selected development cutoffs: {'size_up': None, 'hot_streak': None, 'near_target': None, 'desperate_pace': None}


In [ ]:
# ======================================================
# Stand-down filter: Supporting candidate preparation
# ======================================================

support_with_states = attach_filter_states(
    support_predictions,
    stage1_historical["idea_features"],
    stage1_historical["trades"],
)

candidate_masks = filter_candidate_masks(
    support_with_states
)

print("Available candidate conditions:", list(candidate_masks))

Available candidate conditions: ['size_up', 'hot_streak']


In [ ]:
# ======================================================
# Stand-down filter development: Correlation analysis
# ======================================================

from scipy.stats import spearmanr

CANDIDATE_FEATURES = {
    "size_up": "size_up_ratio",
    "hot_streak": "hot_streak_completed_wins",
    "near_target": "near_target_remaining_profit",
    "desperate_pace": "desperate_pace_profit_per_hour",
}

gated_support = support_with_states.loc[
    support_with_states["raw_fade_decision"]
].copy()

correlation_results = []

for candidate, feature in CANDIDATE_FEATURES.items():
    subset = gated_support[
        [feature, "reverse_profit_per_lot", "person_id"]
    ].replace([np.inf, -np.inf], np.nan).dropna()

    if len(subset) < 3 or subset[feature].nunique() < 2:
        correlation = np.nan
        p_value = np.nan
    else:
        correlation, p_value = spearmanr(
            subset[feature],
            subset["reverse_profit_per_lot"],
        )

    if pd.isna(correlation):
        direction = "Undetermined"
    elif correlation < 0:
        direction = "Higher values → lower RP/lot (>=)"
    elif correlation > 0:
        direction = "Lower values → lower RP/lot (<=)"
    else:
        direction = "No monotonic association"

    correlation_results.append({
        "candidate": candidate,
        "feature": feature,
        "trades": len(subset),
        "persons": subset["person_id"].nunique(),
        "spearman_rho": correlation,
        "naive_p_value": p_value,
        "suggested_direction": direction,
    })

correlation_results = pd.DataFrame(correlation_results)

display(correlation_results.round(4))

,candidate,feature,trades,persons,spearman_rho,naive_p_value,suggested_direction
0,size_up,size_up_ratio,1986,827,-0.0379,0.0916,Higher values → lower RP/lot (>=)
1,hot_streak,hot_streak_completed_wins,1986,827,-0.0641,0.0043,Higher values → lower RP/lot (>=)
2,near_target,near_target_remaining_profit,1986,827,0.1011,0.0000,Lower values → lower RP/lot (<=)
3,desperate_pace,desperate_pace_profit_per_hour,1986,827,0.0805,0.0003,Lower values → lower RP/lot (<=)


## Cluster bootstrap and rule screening

In [ ]:
# ======================================================
# Stand-down filter development: Cluster bootstrap
# ======================================================

# Prespecified cutoff search grid.
# Quantiles are calculated from gated supporting trades only.
CUTOFF_QUANTILES = {
    "size_up": [0.75, 0.85, 0.95],
    "hot_streak": [0.75, 0.85, 0.95],
    "near_target": [0.10, 0.20, 0.30],
    "desperate_pace": [0.75, 0.85, 0.95],
}

CANDIDATE_DIRECTIONS = {
    "size_up": "high",
    "hot_streak": "high",
    "near_target": "low",
    "desperate_pace": "high",
}

# Four candidates × three cutoff quantiles.
NUMBER_OF_COMPARISONS = sum(
    len(quantiles)
    for quantiles in CUTOFF_QUANTILES.values()
)


def edge(data):
    """Calculate lot-weighted reverse profit per lot."""
    if data.empty:
        return np.nan

    total_lots = data["amount"].sum()

    if total_lots <= 0:
        return np.nan

    return float(
        data["reverse_profit"].sum() / total_lots
    )


def person_bootstrap(
    data,
    n_bootstrap=BOOTSTRAP_ITERATIONS,
    seed=SEED,
):
    """Bootstrap lot-weighted RP/lot by resampling entire people."""
    if data.empty or data["person_id"].isna().any():
        return np.array([], dtype=float)

    totals = (
        data.groupby("person_id", sort=True)
        .agg(
            reverse_profit=("reverse_profit", "sum"),
            amount=("amount", "sum"),
        )
        .to_numpy(dtype=float)
    )

    rng = np.random.default_rng(seed)

    sampled_indices = rng.integers(
        0,
        len(totals),
        size=(n_bootstrap, len(totals)),
    )

    sampled_totals = totals[sampled_indices].sum(axis=1)

    return (
        sampled_totals[:, 0]
        / sampled_totals[:, 1]
    )


def bootstrap_ci(samples, confidence=CONFIDENCE_LEVEL):
    """Calculate a two-sided percentile bootstrap confidence interval."""
    if len(samples) == 0:
        return np.nan, np.nan

    alpha = 1 - confidence

    return tuple(
        np.quantile(
            samples,
            [alpha / 2, 1 - alpha / 2],
        )
    )


def build_cutoff_grid(data):
    """Derive candidate cutoffs without using outcome values."""
    cutoff_grid = {}

    for candidate, feature in CANDIDATE_FEATURES.items():
        values = (
            data[feature]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        if candidate in ("near_target", "desperate_pace"):
            values = values.loc[values >= 0]

        if values.empty:
            cutoff_grid[candidate] = []
            continue

        cutoffs = np.quantile(
            values,
            CUTOFF_QUANTILES[candidate],
        )

        if candidate == "hot_streak":
            cutoffs = np.ceil(cutoffs).astype(int)
            cutoffs = cutoffs[cutoffs > 0]

        cutoff_grid[candidate] = sorted(set(cutoffs))

    return cutoff_grid


def choose_stand_down_rules(support):
    """Screen candidate cutoffs using supporting walk-forward predictions."""
    gated = support.loc[
        support["raw_fade_decision"]
    ].copy()

    cutoff_grid = build_cutoff_grid(gated)

    rows = []

    adjusted_confidence = (
        1 - (1 - CONFIDENCE_LEVEL) / NUMBER_OF_COMPARISONS
    )

    for candidate, cutoffs in cutoff_grid.items():
        feature = CANDIDATE_FEATURES[candidate]
        direction = CANDIDATE_DIRECTIONS[candidate]

        for cutoff in cutoffs:
            if direction == "high":
                mask = gated[feature].ge(cutoff)
            else:
                mask = gated[feature].le(cutoff)

            subset = gated.loc[mask.fillna(False)].copy()

            observed_edge = edge(subset)
            n_persons = subset["person_id"].nunique()

            samples = (
                person_bootstrap(subset, seed=SEED)
                if n_persons >= 2
                else np.array([], dtype=float)
            )

            ci_lower, ci_upper = bootstrap_ci(samples)

            adjusted_lower, adjusted_upper = bootstrap_ci(
                samples,
                confidence=adjusted_confidence,
            )

            qualifies = bool(
                len(subset) > 0
                and n_persons >= 2
                and np.isfinite(observed_edge)
                and observed_edge < 0
                and adjusted_upper < 0
            )

            rows.append({
                "candidate": candidate,
                "feature": feature,
                "direction": direction,
                "cutoff": cutoff,
                "trades": len(subset),
                "persons": n_persons,
                "campaigns": subset["campaign_id"].nunique(),
                "coverage_of_gate": (
                    len(subset) / len(gated)
                    if len(gated)
                    else np.nan
                ),
                "lot_weighted_rp_per_lot": observed_edge,
                "ci_95_lower": ci_lower,
                "ci_95_upper": ci_upper,
                "adjusted_ci_lower": adjusted_lower,
                "adjusted_ci_upper": adjusted_upper,
                "qualifies": qualifies,
            })

    screen = pd.DataFrame(rows)

    if screen.empty:
        raise ValueError(
            "No candidate cutoffs are available for screening."
        )

    # Select at most one cutoff per candidate.
    # Choose the least restrictive qualifying cutoff:
    # the one identifying the most gated trades.
    qualifying = (
        screen.loc[screen["qualifies"]]
        .sort_values(
            ["candidate", "trades", "cutoff"],
            ascending=[True, False, True],
        )
        .drop_duplicates("candidate")
    )

    screen["retained"] = False

    screen.loc[
        qualifying.index,
        "retained",
    ] = True

    return screen


# Screen the supporting campaigns only.
filter_screen = choose_stand_down_rules(
    support_with_states
)

selected_rules = filter_screen.loc[
    filter_screen["retained"]
]

RETAINED_FILTERS = tuple(
    selected_rules["candidate"]
)

# Update cutoffs used by filter_candidate_masks().
# Preserve the original numeric defaults for inactive mandatory keys.
cutoff_keys = {
    "size_up": "size_up_ratio",
    "hot_streak": "hot_streak_completed_wins",
    "near_target": "near_target_remaining_profit",
    "desperate_pace": "desperate_pace_profit_per_hour",
}

for row in selected_rules.itertuples():
    FILTER_CUTOFFS[cutoff_keys[row.candidate]] = (
        int(row.cutoff)
        if row.candidate == "hot_streak"
        else float(row.cutoff)
    )

display(
    filter_screen.sort_values(
        ["candidate", "cutoff"]
    ).round(4)
)

print(
    "Retained stand-down rules:",
    RETAINED_FILTERS if RETAINED_FILTERS else "(none)",
)

print(
    "Selected cutoffs:",
    {
        name: FILTER_CUTOFFS[cutoff_keys[name]]
        for name in RETAINED_FILTERS
    },
)

,candidate,feature,direction,cutoff,trades,persons,campaigns,coverage_of_gate,lot_weighted_rp_per_lot,ci_95_lower,ci_95_upper,adjusted_ci_lower,adjusted_ci_upper,qualifies,retained
9,desperate_pace,desperate_pace_profit_per_hour,high,33.8863,497,296,14,0.2503,4.0266,-43.4897,53.2876,-72.9665,81.2057,False,False
10,desperate_pace,desperate_pace_profit_per_hour,high,41.2389,298,187,14,0.1501,-17.6585,-81.6545,41.1657,-109.1742,68.3551,False,False
11,desperate_pace,desperate_pace_profit_per_hour,high,57.7638,100,66,13,0.0504,-62.8311,-179.3268,21.2274,-263.9147,65.6417,False,False
3,hot_streak,hot_streak_completed_wins,high,1.0000,825,451,14,0.4154,-32.4369,-75.6056,10.9472,-96.9330,39.3074,False,False
4,hot_streak,hot_streak_completed_wins,high,2.0000,346,213,14,0.1742,-60.1360,-123.6539,7.8212,-156.2522,37.7167,False,False
5,hot_streak,hot_streak_completed_wins,high,3.0000,141,88,14,0.0710,-42.6612,-125.6565,39.5081,-162.0040,89.7309,False,False
6,near_target,near_target_remaining_profit,low,153.5000,199,118,14,0.1002,-48.5067,-129.2692,42.1281,-154.0589,81.4434,False,False
7,near_target,near_target_remaining_profit,low,262.3200,398,207,14,0.2004,-61.3419,-120.4401,1.0327,-148.0667,35.1175,False,False
8,near_target,near_target_remaining_profit,low,317.5800,596,301,14,0.3001,-35.5379,-87.4369,16.5797,-106.2149,34.3234,False,False
0,size_up,size_up_ratio,high,2.0000,501,346,14,0.2523,-33.2119,-77.1995,10.1662,-98.2581,33.5790,False,False


Retained stand-down rules: (none)
Selected cutoffs: {}


# Pre-registration

In [ ]:
# ======================================================
# Pre-registration
# ======================================================

# The original numerical threshold is valid only for the
# exact Stage 3 fitted model.
STAGE3_REGISTERED_THRESHOLD = 85.102037

# Reconstructed artifacts are saved separately so they are
# never misrepresented as the exact Stage 3 artifacts.
RECONSTRUCTED_MODEL_PATH = (
    ROOT
    / "models"
    / "exp10b_stage4_reconstructed_model.joblib"
)

RECONSTRUCTED_METADATA_PATH = (
    ROOT
    / "models"
    / "exp10b_stage4_reconstructed_metadata.json"
)


# ======================================================
# Prepare campaigns 33-52 continuity training data
# ======================================================

continuity_training_data = historical_features.loc[
    historical_features["campaign_id"].between(
        TRAIN_START,
        TRAIN_END,
        inclusive="both",
    )
].copy()

expected_continuity_campaigns = set(
    range(
        TRAIN_START,
        TRAIN_END + 1,
    )
)

actual_continuity_campaigns = set(
    continuity_training_data[
        "campaign_id"
    ]
    .dropna()
    .astype(int)
    .unique()
)

if actual_continuity_campaigns != expected_continuity_campaigns:
    raise ValueError(
        "Continuity training data must contain exactly "
        f"campaigns {TRAIN_START}-"
        f"{TRAIN_END}. "
        f"Missing: "
        f"{sorted(expected_continuity_campaigns - actual_continuity_campaigns)}. "
        f"Unexpected: "
        f"{sorted(actual_continuity_campaigns - expected_continuity_campaigns)}."
    )

if continuity_training_data.empty:
    raise ValueError(
        "No campaigns 33-52 observations are available "
        "for continuity-model training."
    )

missing_training_features = (
    set(FEATURE_COLUMNS)
    - set(continuity_training_data.columns)
)

if missing_training_features:
    raise ValueError(
        "Continuity training data is missing frozen features: "
        f"{sorted(missing_training_features)}"
    )

if (
    "reverse_profit_per_lot"
    not in continuity_training_data.columns
):
    raise ValueError(
        "Continuity training data does not contain "
        "reverse_profit_per_lot."
    )

if continuity_training_data[
    "reverse_profit_per_lot"
].isna().any():
    raise ValueError(
        "Campaigns 33-52 contain missing "
        "reverse_profit_per_lot training labels."
    )

if continuity_training_data[
    "campaign_id"
].gt(TRAIN_END).any():
    raise AssertionError(
        "Post-campaign-52 observations entered the "
        "continuity training data."
    )


# ======================================================
# Metadata validation helper
# ======================================================

def validate_continuity_metadata(
    metadata,
    require_original_threshold=False,
    require_training_min=False,
):
    """Validate model metadata against the frozen specification."""

    if (
        metadata.get("target")
        != "reverse_profit_per_lot"
    ):
        raise ValueError(
            "Continuity metadata target is not "
            "reverse_profit_per_lot."
        )

    # Feature order matters for prediction.
    if (
        metadata.get("feature_columns")
        != FEATURE_COLUMNS
    ):
        raise ValueError(
            "Continuity metadata feature order differs "
            "from the frozen 22-feature specification."
        )

    if int(
        metadata.get("n_features", -1)
    ) != len(FEATURE_COLUMNS):
        raise ValueError(
            "Continuity metadata feature count is incorrect."
        )

    if require_training_min:
        if int(
            metadata.get("training_campaign_min", -1)
        ) != TRAIN_START:
            raise ValueError(
                "Continuity metadata has an incorrect "
                "first training campaign."
            )

    if int(
        metadata.get("training_campaign_max", -1)
    ) != TRAIN_END:
        raise ValueError(
            "Continuity metadata does not specify training "
            f"through campaign {TRAIN_END}."
        )

    metadata_quantile = metadata.get(
        "selection_quantile"
    )

    if metadata_quantile is None:
        raise ValueError(
            "Continuity metadata does not contain "
            "selection_quantile."
        )

    if not np.isclose(
        float(metadata_quantile),
        QUANTILE,
    ):
        raise ValueError(
            "Continuity metadata selection quantile differs "
            f"from {QUANTILE}."
        )

    metadata_parameters = metadata.get(
        "lightgbm_parameters",
        {},
    )

    for parameter_name, parameter_value in (
        MODEL_PARAMETERS.items()
    ):
        # The original Stage 3 metadata may not contain
        # verbosity, so allow it to be absent.
        if (
            parameter_name == "verbosity"
            and parameter_name not in metadata_parameters
        ):
            continue

        if (
            metadata_parameters.get(parameter_name)
            != parameter_value
        ):
            raise ValueError(
                "Continuity metadata LightGBM parameter "
                f"{parameter_name!r} differs from the "
                "frozen model specification."
            )

    if require_original_threshold:
        metadata_threshold = metadata.get(
            "decision_threshold"
        )

        if metadata_threshold is None:
            raise ValueError(
                "Original metadata does not contain "
                "decision_threshold."
            )

        if not np.isclose(
            float(metadata_threshold),
            STAGE3_REGISTERED_THRESHOLD,
        ):
            raise ValueError(
                "Original Stage 3 threshold differs from "
                f"{STAGE3_REGISTERED_THRESHOLD}."
            )


# ======================================================
# Identify available artifact pairs
# ======================================================

exact_original_available = (
    ORIGINAL_MODEL_PATH.is_file()
    and ORIGINAL_METADATA_PATH.is_file()
)

reconstructed_artifacts_available = (
    RECONSTRUCTED_MODEL_PATH.is_file()
    and RECONSTRUCTED_METADATA_PATH.is_file()
)


# ======================================================
# Case 1: Load the exact Stage 3 model
# ======================================================

if exact_original_available:
    original_metadata = json.loads(
        ORIGINAL_METADATA_PATH.read_text(
            encoding="utf-8"
        )
    )

    if original_metadata.get("experiment") != "exp10b":
        raise ValueError(
            "Original metadata does not describe "
            "Experiment 10B."
        )

    validate_continuity_metadata(
        metadata=original_metadata,
        require_original_threshold=True,
        require_training_min=False,
    )

    original_model = joblib.load(
        ORIGINAL_MODEL_PATH
    )

    # Retain the exact Stage 3 numerical threshold only when
    # the exact Stage 3 fitted model is available.
    ORIGINAL_THRESHOLD = (
        STAGE3_REGISTERED_THRESHOLD
    )

    continuity_model_source = (
        "exact_stage3_artifact"
    )

    continuity_model_path_used = (
        ORIGINAL_MODEL_PATH
    )

    continuity_metadata_path_used = (
        ORIGINAL_METADATA_PATH
    )

    print(
        "Loaded exact Stage 3 continuity model:",
        ORIGINAL_MODEL_PATH,
    )


# ======================================================
# Case 2: Load an existing Stage 4 reconstruction
# ======================================================

elif reconstructed_artifacts_available:
    reconstructed_metadata = json.loads(
        RECONSTRUCTED_METADATA_PATH.read_text(
            encoding="utf-8"
        )
    )

    if (
        reconstructed_metadata.get("artifact_type")
        != "stage4_reconstruction"
    ):
        raise ValueError(
            "Reconstructed metadata has an unexpected "
            "artifact type."
        )

    validate_continuity_metadata(
        metadata=reconstructed_metadata,
        require_original_threshold=False,
        require_training_min=True,
    )

    reconstructed_threshold = float(
        reconstructed_metadata.get(
            "decision_threshold",
            np.nan,
        )
    )

    if not np.isfinite(
        reconstructed_threshold
    ):
        raise ValueError(
            "Reconstructed metadata contains an invalid "
            "decision threshold."
        )

    original_model = joblib.load(
        RECONSTRUCTED_MODEL_PATH
    )

    ORIGINAL_THRESHOLD = (
        reconstructed_threshold
    )

    continuity_model_source = (
        "saved_stage4_reconstruction_campaigns_33_52"
    )

    continuity_model_path_used = (
        RECONSTRUCTED_MODEL_PATH
    )

    continuity_metadata_path_used = (
        RECONSTRUCTED_METADATA_PATH
    )

    print(
        "Loaded previously reconstructed continuity model:",
        RECONSTRUCTED_MODEL_PATH,
    )

    print(
        "Loaded reconstructed threshold:",
        f"{ORIGINAL_THRESHOLD:.6f}",
    )


# ======================================================
# Case 3: Train a new reconstruction on campaigns 33-52
# ======================================================

else:
    # Do not use incomplete artifact pairs.
    if (
        ORIGINAL_MODEL_PATH.is_file()
        != ORIGINAL_METADATA_PATH.is_file()
    ):
        print(
            "Warning: only one original Stage 3 artifact "
            "exists. The incomplete pair will not be used."
        )

    if (
        RECONSTRUCTED_MODEL_PATH.is_file()
        != RECONSTRUCTED_METADATA_PATH.is_file()
    ):
        print(
            "Warning: only one reconstructed artifact "
            "exists. A new reconstruction will be trained."
        )

    print(
        "No complete continuity-model artifact was found."
    )

    print(
        "Training a new continuity model using campaigns "
        "33-52 and the corrected Stage 4 feature pipeline..."
    )

    original_model = fit_fixed_regressor(
        continuity_training_data
    )

    continuity_training_scores = np.asarray(
        original_model.predict(
            continuity_training_data[
                FEATURE_COLUMNS
            ]
        ),
        dtype=float,
    )

    if (
        len(continuity_training_scores)
        != len(continuity_training_data)
    ):
        raise AssertionError(
            "Reconstructed-model prediction count does "
            "not match the campaigns 33-52 training rows."
        )

    if not np.isfinite(
        continuity_training_scores
    ).all():
        raise ValueError(
            "Reconstructed continuity model produced "
            "non-finite training predictions."
        )

    # Newly fitted weights require a newly calibrated
    # threshold. Preserve the registered q90 rule instead
    # of incorrectly reusing the numerical value 85.102037.
    ORIGINAL_THRESHOLD = float(
        np.quantile(
            continuity_training_scores,
            QUANTILE,
        )
    )

    continuity_model_source = (
        "new_stage4_reconstruction_campaigns_33_52"
    )

    continuity_model_path_used = (
        RECONSTRUCTED_MODEL_PATH
    )

    continuity_metadata_path_used = (
        RECONSTRUCTED_METADATA_PATH
    )

    reconstructed_metadata = {
        "artifact_type": "stage4_reconstruction",
        "experiment": "exp10b_stage4_reconstruction",
        "source_experiment": "exp10b",
        "target": "reverse_profit_per_lot",
        "feature_columns": FEATURE_COLUMNS,
        "n_features": len(FEATURE_COLUMNS),

        "training_campaign_min": (
            TRAIN_START
        ),
        "training_campaign_max": (
            TRAIN_END
        ),
        "training_campaigns": list(
            range(
                TRAIN_START,
                TRAIN_END + 1,
            )
        ),
        "training_rows": int(
            len(continuity_training_data)
        ),

        "selection_quantile": QUANTILE,
        "decision_threshold": (
            ORIGINAL_THRESHOLD
        ),
        "original_stage3_threshold": (
            STAGE3_REGISTERED_THRESHOLD
        ),

        "lightgbm_parameters": (
            MODEL_PARAMETERS
        ),

        "feature_pipeline": (
            "Corrected Stage 4 person-keyed "
            "feature pipeline"
        ),

        "person_key": (
            "SHA256 hash of normalized email after joining "
            "within campaign on campaign_id and account_id"
        ),
    }

    RECONSTRUCTED_MODEL_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    joblib.dump(
        original_model,
        RECONSTRUCTED_MODEL_PATH,
    )

    RECONSTRUCTED_METADATA_PATH.write_text(
        json.dumps(
            reconstructed_metadata,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        "Saved reconstructed model:",
        RECONSTRUCTED_MODEL_PATH,
    )

    print(
        "Saved reconstructed metadata:",
        RECONSTRUCTED_METADATA_PATH,
    )

    print(
        "Recalibrated q90 threshold:",
        f"{ORIGINAL_THRESHOLD:.6f}",
    )


# ======================================================
# Validate model compatibility
# ======================================================

if not hasattr(original_model, "predict"):
    raise TypeError(
        "Continuity model does not support prediction."
    )

model_feature_count = getattr(
    original_model,
    "n_features_in_",
    None,
)

if (
    model_feature_count is not None
    and int(model_feature_count)
    != len(FEATURE_COLUMNS)
):
    raise ValueError(
        "Continuity model expects "
        f"{model_feature_count} features, but the frozen "
        f"specification contains {len(FEATURE_COLUMNS)}."
    )

# Run a small prediction before opening any sealed data.
compatibility_sample = (
    continuity_training_data[
        FEATURE_COLUMNS
    ]
    .head(
        min(
            10,
            len(continuity_training_data),
        )
    )
)

compatibility_predictions = np.asarray(
    original_model.predict(
        compatibility_sample
    ),
    dtype=float,
)

if (
    len(compatibility_predictions)
    != len(compatibility_sample)
):
    raise AssertionError(
        "Continuity-model compatibility prediction "
        "returned an unexpected number of rows."
    )

if not np.isfinite(
    compatibility_predictions
).all():
    raise ValueError(
        "Continuity model produced non-finite values "
        "during the compatibility check."
    )

print(
    "Continuity model ready."
)

print(
    "Continuity model source:",
    continuity_model_source,
)

print(
    "Effective continuity threshold:",
    f"{ORIGINAL_THRESHOLD:.6f}",
)


# ======================================================
# Register evaluated strategies
# ======================================================

# Keep these keys unchanged because the later strategy
# evaluation and metrics cells expect these exact names.
REGISTERED_STRATEGIES = {
    "walk_forward_gate": (
        "Expanding-window LightGBM refit; "
        "training-score 90th-percentile gate; PRIMARY"
    ),
    "original_continuity": (
        (
            "Exact original campaigns 33-52 Stage 3 fitted "
            "model; fixed numerical threshold 85.102037"
        )
        if continuity_model_source
        == "exact_stage3_artifact"
        else (
            "Stage 4 reconstruction fitted on campaigns "
            "33-52 using the corrected person-keyed feature "
            "pipeline; training-score 90th-percentile gate"
        )
    ),
}


# ======================================================
# Determine genuinely eligible sealed campaigns
# ======================================================

invalid_exposed_campaigns = (
    set(EXPOSED_CAMPAIGNS)
    - set(SEALED_CAMPAIGNS)
)

if invalid_exposed_campaigns:
    raise ValueError(
        "Exposed campaigns must belong to the planned "
        "sealed range. Invalid campaigns: "
        f"{sorted(invalid_exposed_campaigns)}"
    )

SEALED_ELIGIBLE = tuple(
    campaign_id
    for campaign_id in SEALED_CAMPAIGNS
    if campaign_id not in EXPOSED_CAMPAIGNS
)

if not SEALED_ELIGIBLE:
    raise ValueError(
        "No eligible sealed campaigns remain."
    )


# ======================================================
# Record the frozen evaluation protocol
# ======================================================

registration = {
    "registered_strategies": (
        REGISTERED_STRATEGIES
    ),

    "primary_strategy": (
        "walk_forward_gate"
    ),

    "continuity_strategy": (
        "original_continuity"
    ),

    "strategy_excluded_before_sealed_evaluation": {
        "walk_forward_with_filter": (
            "Dropped before sealed evaluation because no "
            "candidate stand-down rule had negative "
            "lot-weighted RP/lot with its adjusted confidence "
            "interval entirely below zero."
        )
    },

    "first_training_campaign": (
        TRAIN_START
    ),

    "initial_training_end": (
        TRAIN_END
    ),

    "supporting_campaigns": list(
        range(
            SUPPORT_START,
            SUPPORT_END + 1,
        )
    ),

    "supporting_data_role": (
        "Supporting evidence only; not used for the final "
        "Stage 4 pass/fail decision."
    ),

    "sealed_eligible_campaigns": list(
        SEALED_ELIGIBLE
    ),

    "exposed_campaigns": list(
        EXPOSED_CAMPAIGNS
    ),

    "frozen_features": (
        FEATURE_COLUMNS
    ),

    "model_parameters": (
        MODEL_PARAMETERS
    ),

    "model_target": (
        "reverse_profit_per_lot"
    ),

    "walk_forward_training_rule": (
        "For each evaluated campaign, refit the frozen model "
        "on all campaigns strictly earlier than that campaign."
    ),

    "quantile": (
        QUANTILE
    ),

    "threshold_calibration": (
        "90th percentile of predictions on the refitted "
        "model's training observations."
    ),

    # --------------------------------------------------
    # Continuity-model provenance
    # --------------------------------------------------

    "continuity_model_source": (
        continuity_model_source
    ),

    "continuity_training_campaigns": list(
        range(
            TRAIN_START,
            TRAIN_END + 1,
        )
    ),

    "continuity_training_rows": int(
        len(continuity_training_data)
    ),

    "stage3_registered_threshold": (
        STAGE3_REGISTERED_THRESHOLD
    ),

    "continuity_decision_threshold": float(
        ORIGINAL_THRESHOLD
    ),

    "continuity_threshold_rule": (
        (
            "Fixed original Stage 3 numerical threshold."
        )
        if continuity_model_source
        == "exact_stage3_artifact"
        else (
            "Recalibrated 90th percentile of predictions "
            "on campaigns 33-52 because the exact Stage 3 "
            "fitted weights were unavailable."
        )
    ),

    "continuity_model_path_used": str(
        continuity_model_path_used
    ),

    "continuity_metadata_path_used": str(
        continuity_metadata_path_used
    ),

    "original_model_path": str(
        ORIGINAL_MODEL_PATH
    ),

    "original_metadata_path": str(
        ORIGINAL_METADATA_PATH
    ),

    # --------------------------------------------------
    # Identity specification
    # --------------------------------------------------

    "person_key": (
        "SHA256 hash of normalized email. Email is obtained "
        "by joining User Trades to User Data on campaign_id "
        "and account_id within each campaign. Confirmed by C22."
    ),

    "missing_email_policy": (
        MISSING_EMAIL_POLICY
    ),

    # --------------------------------------------------
    # Metrics and statistical inference
    # --------------------------------------------------

    "primary_metric": (
        "lot_weighted_reverse_profit_per_lot"
    ),

    "coverage_definition": (
        "number of faded trades divided by all eligible trades"
    ),

    "alpha": 0.05,

    "inference": (
        "Person-cluster bootstrap with Holm correction "
        "across the two registered strategy tests."
    ),

    "multiple_testing_method": (
        "holm"
    ),

    "number_of_registered_tests": len(
        REGISTERED_STRATEGIES
    ),

    "bootstrap_iterations": (
        BOOTSTRAP_ITERATIONS
    ),

    # --------------------------------------------------
    # Primary pass bar
    # --------------------------------------------------

    "positive_campaign_share_minimum": (
        MIN_POSITIVE_CAMPAIGN_SHARE
    ),

    "primary_edge_must_be_positive": True,

    "primary_ci_lower_must_exceed_zero": True,

    "primary_pass_bar": {
        "lot_weighted_rp_per_lot_above_zero": True,
        "confidence_interval_lower_bound_above_zero": True,
        "minimum_positive_campaign_share": (
            MIN_POSITIVE_CAMPAIGN_SHARE
        ),
    },

    "sealed_evaluation_status": (
        "LOCKED"
    ),
}


# ======================================================
# Review and save the pre-registration
# ======================================================

print(
    "Registered sealed campaigns:",
    registration[
        "sealed_eligible_campaigns"
    ],
)

print(
    "Registered strategies:",
    list(REGISTERED_STRATEGIES),
)

print(
    "Continuity model source:",
    continuity_model_source,
)

print(
    "Continuity decision threshold:",
    f"{ORIGINAL_THRESHOLD:.6f}",
)

print(
    "Primary pass bar: lot-weighted RP/lot > 0; "
    "95% CI lower bound > 0; "
    f"at least "
    f"{MIN_POSITIVE_CAMPAIGN_SHARE:.0%} "
    "positive sealed campaigns."
)

print(
    "Person identity: C22-confirmed normalized email."
)

print(
    "Stand-down filter: excluded before sealed evaluation."
)

print(
    "Review this registration before setting "
    "PROTOCOL_APPROVED=True and "
    "RUN_SEALED_EVALUATION=True."
)


# ======================================================
# Save protocol records
# ======================================================

# Saving these files does not read or evaluate sealed data.
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

registration_path = (
    OUTPUT_DIR
    / "preregistration.json"
)

registration_path.write_text(
    json.dumps(
        registration,
        indent=2,
    ),
    encoding="utf-8",
)

identity_audit.to_csv(
    OUTPUT_DIR
    / "identity_audit.csv",
    index=False,
)

print(
    "Saved pre-registration:",
    registration_path,
)

No complete continuity-model artifact was found.
Training a new continuity model using campaigns 33-52 and the corrected Stage 4 feature pipeline...
Saved reconstructed model: /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/models/exp10b_stage4_reconstructed_model.joblib
Saved reconstructed metadata: /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/models/exp10b_stage4_reconstructed_metadata.json
Recalibrated q90 threshold: 60.439365
Continuity model ready.
Continuity model source: new_stage4_reconstruction_campaigns_33_52
Effective continuity threshold: 60.439365
Registered sealed campaigns: [67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82]
Registered strategies: ['walk_forward_gate', 'original_continuity']
Continuity model source: new_stage4_reconstruction_campaigns_33_52
Continuity decision threshold: 60.439365
Primary pass bar: lot-weighted RP/lot > 0; 95% CI lower bound > 0; at least 60% positive sea

# Sealed evaluation

## Locked input loading

In [ ]:
# ======================================================
# Sealed evaluation: Locked input loading
# ======================================================

SEALED_TRADES_DIR = (
    SEALED_DIR
    / "User Trades"
)

SEALED_USERS_DIR = (
    SEALED_DIR
    / "User Data"
)


def build_sealed_identity_table(
    users,
    trades,
):
    """
    Build a campaign-specific account-to-person mapping for sealed data.

    Trades are joined to User Data using campaign_id and account_id.
    Normalized email is then hashed to create the person identifier.
    Missing emails use a campaign-specific fallback identifier.
    """
    users = users.copy()
    trades = trades.copy()

    join_columns = [
        "campaign_id",
        "account_id",
    ]

    required_user_columns = {
        "campaign_id",
        "account_id",
        "email",
    }

    required_trade_columns = {
        "campaign_id",
        "account_id",
    }

    missing_user_columns = (
        required_user_columns
        - set(users.columns)
    )

    missing_trade_columns = (
        required_trade_columns
        - set(trades.columns)
    )

    if missing_user_columns:
        raise ValueError(
            "Sealed User Data is missing columns: "
            f"{sorted(missing_user_columns)}"
        )

    if missing_trade_columns:
        raise ValueError(
            "Sealed trade data is missing columns: "
            f"{sorted(missing_trade_columns)}"
        )

    # Normalize the campaign-account join keys.
    users["campaign_id"] = (
        users["campaign_id"].astype(int)
    )

    trades["campaign_id"] = (
        trades["campaign_id"].astype(int)
    )

    users["account_id"] = normalise_account(
        users["account_id"]
    )

    trades["account_id"] = normalise_account(
        trades["account_id"]
    )

    users["email"] = normalise_email(
        users["email"]
    )

    if trades["account_id"].isna().any():
        raise ValueError(
            "Sealed trades contain missing account IDs."
        )

    # Check whether one campaign-account pair maps to
    # more than one non-missing email.
    email_counts = (
        users.groupby(join_columns)["email"]
        .nunique(dropna=True)
    )

    conflicting_mappings = int(
        email_counts.gt(1).sum()
    )

    if conflicting_mappings:
        raise ValueError(
            f"{conflicting_mappings} sealed campaign-account "
            "pairs map to multiple emails."
        )

    # If duplicate rows agree, retain one mapping.
    users = (
        users[
            join_columns + ["email"]
        ]
        .drop_duplicates()
    )

    # Reject a campaign-account pair that has both a
    # missing-email row and a non-missing-email row.
    email_presence_counts = (
        users.assign(
            email_missing=users["email"].isna()
        )
        .groupby(join_columns)["email_missing"]
        .nunique()
    )

    inconsistent_mappings = int(
        email_presence_counts.gt(1).sum()
    )

    if inconsistent_mappings:
        raise ValueError(
            f"{inconsistent_mappings} sealed campaign-account "
            "pairs contain both missing and present emails."
        )

    users = (
        users.drop_duplicates(
            subset=join_columns
        )
    )

    trading_accounts = (
        trades[join_columns]
        .drop_duplicates()
    )

    # Preserve every account that actually traded,
    # including accounts missing from User Data.
    mapping = trading_accounts.merge(
        users,
        on=join_columns,
        how="left",
        indicator=True,
        validate="one_to_one",
    )

    missing_user_record = (
        mapping["_merge"].eq("left_only")
    )

    missing_email = (
        mapping["_merge"].eq("both")
        & mapping["email"].isna()
    )

    email_available = (
        mapping["_merge"].eq("both")
        & mapping["email"].notna()
    )

    mapping["identity_status"] = (
        "missing_user_record_fallback"
    )

    mapping.loc[
        missing_email,
        "identity_status",
    ] = "missing_email_fallback"

    mapping.loc[
        email_available,
        "identity_status",
    ] = "email_based"

    mapping["person_id"] = [
        make_person_id(
            email=row.email,
            campaign_id=row.campaign_id,
            account_id=row.account_id,
        )
        for row in mapping.itertuples(
            index=False
        )
    ]

    if mapping["person_id"].isna().any():
        raise AssertionError(
            "Some sealed trading accounts have no person ID."
        )

    identity_table = mapping[
        join_columns
        + [
            "person_id",
            "identity_status",
        ]
    ].copy()

    identity_summary = (
        identity_table[
            "identity_status"
        ]
        .value_counts()
        .rename_axis("identity_status")
        .reset_index(name="account_mappings")
    )

    print("\nSealed identity summary:")
    display(identity_summary)

    return identity_table


# ======================================================
# Keep sealed inputs locked until both conditions are met
# ======================================================

if not RUN_SEALED_EVALUATION:
    print(
        "SEALED DATA LOCKED. "
        "Set RUN_SEALED_EVALUATION=True only after "
        "finalising and saving the protocol."
    )

else:
    # Approval is checked before any sealed file is read.
    if not PROTOCOL_APPROVED:
        raise RuntimeError(
            "Review and freeze the pre-registration, then set "
            "PROTOCOL_APPROVED=True before opening sealed data."
        )

    print(
        "Opening sealed trade and User Data files for "
        "the first registered evaluation..."
    )

    if not SEALED_TRADES_DIR.is_dir():
        raise FileNotFoundError(
            "Sealed trade directory not found: "
            f"{SEALED_TRADES_DIR}"
        )

    if not SEALED_USERS_DIR.is_dir():
        raise FileNotFoundError(
            "Sealed User Data directory not found: "
            f"{SEALED_USERS_DIR}"
        )

    # --------------------------------------------------
    # 1. Load sealed campaigns
    # --------------------------------------------------

    sealed_trades_raw = load_trade_directory(
        SEALED_TRADES_DIR,
        is_unseen=True,
    )

    sealed_users_raw = load_user_directory(
        SEALED_USERS_DIR
    )

    # --------------------------------------------------
    # 2. Validate campaign coverage
    # --------------------------------------------------

    found_trade_campaigns = set(
        sealed_trades_raw[
            "campaign_id"
        ].astype(int).unique()
    )

    found_user_campaigns = set(
        sealed_users_raw[
            "campaign_id"
        ].astype(int).unique()
    )

    expected_sealed_campaigns = set(
        SEALED_CAMPAIGNS
    )

    if found_trade_campaigns != expected_sealed_campaigns:
        raise ValueError(
            "Sealed trade campaign mismatch. "
            f"Found {sorted(found_trade_campaigns)}; "
            f"expected {sorted(expected_sealed_campaigns)}."
        )

    if found_user_campaigns != expected_sealed_campaigns:
        raise ValueError(
            "Sealed User Data campaign mismatch. "
            f"Found {sorted(found_user_campaigns)}; "
            f"expected {sorted(expected_sealed_campaigns)}."
        )

    # Remove any campaign that was declared exposed
    # before pre-registration.
    sealed_trades_raw = (
        sealed_trades_raw.loc[
            sealed_trades_raw[
                "campaign_id"
            ].isin(SEALED_ELIGIBLE)
        ]
        .copy()
    )

    sealed_users_raw = (
        sealed_users_raw.loc[
            sealed_users_raw[
                "campaign_id"
            ].isin(SEALED_ELIGIBLE)
        ]
        .copy()
    )

    # --------------------------------------------------
    # 3. Validate required sealed outcomes
    # --------------------------------------------------

    if sealed_trades_raw[
        "close_date_time"
    ].isna().any():
        raise ValueError(
            "Sealed trades contain missing close timestamps. "
            "Close timestamps are required for trade-idea "
            "grouping and sequential feature construction."
        )

    if sealed_trades_raw[
        "net_profit"
    ].isna().any():
        raise ValueError(
            "Sealed trades contain missing net_profit. "
            "It is required for later-campaign historical "
            "state and past-idea features."
        )

    if sealed_trades_raw[
        "reverse_profit"
    ].isna().any():
        raise ValueError(
            "Sealed trades contain missing reverse_profit. "
            "It is required for final strategy evaluation."
        )

    if (
        sealed_trades_raw["amount"].isna().any()
        or sealed_trades_raw["amount"].le(0).any()
    ):
        raise ValueError(
            "Sealed trades contain missing or non-positive amounts."
        )

    # --------------------------------------------------
    # 4. Create and attach person identities
    # --------------------------------------------------

    sealed_identity_table = (
        build_sealed_identity_table(
            users=sealed_users_raw,
            trades=sealed_trades_raw,
        )
    )

    sealed_trades = attach_person_ids(
        trades=sealed_trades_raw,
        identity_table=sealed_identity_table,
    )

    if sealed_trades["person_id"].isna().any():
        raise AssertionError(
            "Some sealed trades remain without person IDs."
        )

    # --------------------------------------------------
    # 5. Combine historical and sealed trades
    # --------------------------------------------------

    combined_trades = pd.concat(
        [
            historical_trades,
            sealed_trades,
        ],
        ignore_index=True,
        sort=False,
    )

    expected_combined_campaigns = (
        set(
            range(
                TRAIN_START,
                SUPPORT_END + 1,
            )
        )
        | set(SEALED_ELIGIBLE)
    )

    actual_combined_campaigns = set(
        combined_trades[
            "campaign_id"
        ].astype(int).unique()
    )

    if actual_combined_campaigns != expected_combined_campaigns:
        raise ValueError(
            "Combined campaign mismatch. "
            f"Found {sorted(actual_combined_campaigns)}; "
            f"expected {sorted(expected_combined_campaigns)}."
        )

    # --------------------------------------------------
    # 6. Reconstruct sequential features
    # --------------------------------------------------

    stage1_all = build_stage1_tables_from_trades(
        combined_trades
    )

    full_features = build_model_features(
        stage1_all
    )

    if len(full_features) != len(combined_trades):
        raise AssertionError(
            "Feature engineering changed the number of trade rows."
        )

    missing_features = (
        set(FEATURE_COLUMNS)
        - set(full_features.columns)
    )

    if missing_features:
        raise AssertionError(
            "Sealed feature table is missing frozen features: "
            f"{sorted(missing_features)}"
        )

    if full_features[
        "person_id"
    ].isna().any():
        raise AssertionError(
            "Feature table contains missing person IDs."
        )

    # The continuity strategy uses the original Stage 3
    # model and threshold, but the corrected Stage 4
    # person-keyed feature table.
    continuity_features = full_features

    print(
        "\nSealed feature construction completed."
    )

    print(
        "Combined trade rows:",
        f"{len(combined_trades):,}",
    )

    print(
        "Full feature rows:",
        f"{len(full_features):,}",
    )

    print(
        "Eligible sealed campaigns:",
        list(SEALED_ELIGIBLE),
    )

Opening sealed trade and User Data files for the first registered evaluation...

Scanning unseen trade directory:
  /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/data/sealed/User Trades
Found 16 unseen files.
  [1/16] Processing: Campaign 67 Data 07 July 2026 XAUUSD only (1D).csv
  [2/16] Processing: Campaign 68 Data 10 July 2026 XAUUSD only (1D).csv
  [3/16] Processing: Campaign 69 Data 14 July 2026 XAUUSD only (1D).csv
  [4/16] Processing: Campaign 70 Data 17 July 2026 XAUUSD only (1D).csv
  [5/16] Processing: Campaign 71 Data 21 Jul 2026 XAUUSD only (1D).csv
  [6/16] Processing: Campaign 72 Data 24 Jul 2026 XAUUSD only (1D).csv
  [7/16] Processing: Campaign 73 Data 28 Jul 2026 XAUUSD only (1D).csv
  [8/16] Processing: Campaign 74 Data 31 Jul 2026 XAUUSD only (1D).csv
  [9/16] Processing: Campaign 75 Data 04 Aug 2026 XAUUSD only (1D).csv
  [10/16] Processing: Campaign 76 Data 07 Aug 2026 XAUUSD only (1D).csv
  [11/16] Processing: Campaign 77 Data 1

,identity_status,account_mappings
0,email_based,4743
1,missing_user_record_fallback,3


Person IDs attached successfully: 31,498 trades.

Sealed feature construction completed.
Combined trade rows: 78,018
Full feature rows: 78,018
Eligible sealed campaigns: [67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82]


## Two registered strategies

In [ ]:
# ======================================================
# Sealed evaluation: Two registered strategies
# ======================================================

if not RUN_SEALED_EVALUATION:
    print(
        "Skipped: sealed evaluation is locked."
    )

else:
    expected_strategies = {
        "walk_forward_gate",
        "original_continuity",
    }

    registered_strategy_names = set(
        REGISTERED_STRATEGIES
    )

    if registered_strategy_names != expected_strategies:
        raise ValueError(
            "Registered strategies do not match the frozen "
            "two-strategy protocol. "
            f"Found {sorted(registered_strategy_names)}; "
            f"expected {sorted(expected_strategies)}."
        )

    all_decisions = []

    for campaign_id in SEALED_ELIGIBLE:
        print(
            f"\nCampaign {campaign_id}: "
            "evaluating two registered strategies..."
        )

        # --------------------------------------------------
        # 1. Primary expanding-window walk-forward strategy
        # --------------------------------------------------

        # This function:
        # - trains on campaigns strictly before campaign_id;
        # - refits the frozen LightGBM configuration;
        # - calculates the 90th-percentile training-score gate;
        # - scores the current campaign.
        current = fit_and_score_campaign(
            feature_data=full_features,
            campaign_id=campaign_id,
        )

        if current.empty:
            raise ValueError(
                f"No feature rows found for campaign {campaign_id}."
            )

        if current["trade_row_id"].duplicated().any():
            raise AssertionError(
                f"Campaign {campaign_id} contains duplicate "
                "trade_row_id values."
            )

        if current["person_id"].isna().any():
            raise AssertionError(
                f"Campaign {campaign_id} contains missing person IDs."
            )

        if current[FEATURE_COLUMNS].shape[1] != len(
            FEATURE_COLUMNS
        ):
            raise AssertionError(
                "Primary model feature count is inconsistent."
            )

        # --------------------------------------------------
        # 2. Original Stage 3 continuity strategy
        # --------------------------------------------------

        # Use the original campaigns 33-52 model weights and
        # the fixed Stage 3 threshold. The features themselves
        # come from the corrected Stage 4 person-keyed pipeline.
        continuity_scores = continuity_model.predict(
            current[FEATURE_COLUMNS]
        )

        continuity_scores = np.asarray(
            continuity_scores,
            dtype=float,
        )

        if len(continuity_scores) != len(current):
            raise AssertionError(
                f"Campaign {campaign_id}: continuity prediction "
                "count does not match the trade count."
            )

        if not np.isfinite(continuity_scores).all():
            raise ValueError(
                f"Campaign {campaign_id}: continuity model "
                "produced non-finite scores."
            )

        current = current.copy()

        current["continuity_score"] = (
            continuity_scores
        )

        # --------------------------------------------------
        # 3. Create trade-level decisions for each strategy
        # --------------------------------------------------

        for strategy_name in REGISTERED_STRATEGIES:
            if strategy_name == "walk_forward_gate":
                scores = current[
                    "predicted_score"
                ]

                thresholds = current[
                    "decision_threshold"
                ]

                fade_decisions = current[
                    "raw_fade_decision"
                ]

            elif strategy_name == "original_continuity":
                scores = current[
                    "continuity_score"
                ]

                thresholds = pd.Series(
                    CONTINUITY_THRESHOLD,
                    index=current.index,
                    dtype=float,
                )

                fade_decisions = (
                    current["continuity_score"]
                    .ge(CONTINUITY_THRESHOLD)
                )

            else:
                raise ValueError(
                    "Unexpected registered strategy: "
                    f"{strategy_name}"
                )

            frame = current[
                [
                    "trade_row_id",
                    "campaign_id",
                    "account_id",
                    "person_id",
                    "amount",
                    "reverse_profit",
                    "open_date_time",
                ]
            ].copy()

            frame["strategy"] = (
                strategy_name
            )

            frame["score"] = np.asarray(
                scores,
                dtype=float,
            )

            frame["threshold"] = np.asarray(
                thresholds,
                dtype=float,
            )

            frame["fade_decision"] = np.asarray(
                fade_decisions,
                dtype=bool,
            )

            all_decisions.append(
                frame
            )

    # ------------------------------------------------------
    # 4. Combine and validate all trade-level decisions
    # ------------------------------------------------------

    if not all_decisions:
        raise ValueError(
            "No sealed strategy decisions were generated."
        )

    sealed_decisions = pd.concat(
        all_decisions,
        ignore_index=True,
    )

    if sealed_decisions.duplicated(
        subset=[
            "strategy",
            "trade_row_id",
        ]
    ).any():
        raise AssertionError(
            "Duplicate strategy-trade decisions were generated."
        )

    strategy_row_counts = (
        sealed_decisions.groupby(
            "strategy"
        )
        .size()
    )

    if strategy_row_counts.nunique() != 1:
        raise AssertionError(
            "Registered strategies were evaluated on "
            "different numbers of trades."
        )

    # Confirm that both strategies evaluated exactly the
    # same set of trade rows.
    strategy_trade_sets = {
        strategy_name: set(
            strategy_data["trade_row_id"]
        )
        for strategy_name, strategy_data
        in sealed_decisions.groupby("strategy")
    }

    reference_trade_set = next(
        iter(strategy_trade_sets.values())
    )

    for strategy_name, trade_set in (
        strategy_trade_sets.items()
    ):
        if trade_set != reference_trade_set:
            raise AssertionError(
                f"{strategy_name} was evaluated on a "
                "different trade population."
            )

    expected_decision_rows = (
        len(reference_trade_set)
        * len(REGISTERED_STRATEGIES)
    )

    if len(sealed_decisions) != expected_decision_rows:
        raise AssertionError(
            "Unexpected number of strategy/trade decision rows."
        )

    print(
        f"\nGenerated {len(sealed_decisions):,} "
        "strategy/trade decision rows."
    )

    print("\nDecision rows per strategy:")
    display(
        strategy_row_counts
        .rename("decision_rows")
        .to_frame()
    )

    print("\nCoverage by campaign and strategy:")
    display(
        sealed_decisions
        .groupby(
            [
                "campaign_id",
                "strategy",
            ]
        )
        .agg(
            trades_evaluated=(
                "trade_row_id",
                "size",
            ),
            trades_faded=(
                "fade_decision",
                "sum",
            ),
            coverage=(
                "fade_decision",
                "mean",
            ),
        )
        .reset_index()
    )


Campaign 67: evaluating two registered strategies...
Campaign 67: trained=46,520, scored=1,673, threshold=31.341518, coverage=8.07%


NameError: name 'continuity_model' is not defined

## Metrics and Holm correction

In [ ]:
# ======================================================
# Metrics and Holm correction
# ======================================================


def lot_weighted_edge(data):
    """
    Calculate total reverse profit divided by total lots.

    This is the required lot-weighted reverse profit per lot,
    rather than the unweighted mean of trade-level RP/lot.
    """
    if data.empty:
        return np.nan

    total_lots = data["amount"].sum()

    if (
        not np.isfinite(total_lots)
        or total_lots <= 0
    ):
        return np.nan

    return float(
        data["reverse_profit"].sum()
        / total_lots
    )


def person_cluster_bootstrap(
    data,
    n_bootstrap=BOOTSTRAP_ITERATIONS,
    seed=SEED,
):
    """
    Bootstrap lot-weighted RP/lot by resampling entire people.

    All selected trades belonging to one person remain together
    within a bootstrap sample.
    """
    if data.empty:
        return np.array(
            [],
            dtype=float,
        )

    required_columns = {
        "person_id",
        "reverse_profit",
        "amount",
    }

    missing_columns = (
        required_columns
        - set(data.columns)
    )

    if missing_columns:
        raise ValueError(
            "Bootstrap data is missing columns: "
            f"{sorted(missing_columns)}"
        )

    if data[
        list(required_columns)
    ].isna().any().any():
        raise ValueError(
            "Bootstrap data contains missing person IDs, "
            "reverse-profit values, or trade amounts."
        )

    person_totals = (
        data.groupby(
            "person_id",
            sort=True,
        )
        .agg(
            reverse_profit=(
                "reverse_profit",
                "sum",
            ),
            amount=(
                "amount",
                "sum",
            ),
        )
        .reset_index(drop=True)
    )

    if person_totals.empty:
        return np.array(
            [],
            dtype=float,
        )

    if person_totals["amount"].le(0).any():
        raise ValueError(
            "Person-level bootstrap totals contain "
            "non-positive trade amounts."
        )

    values = person_totals[
        [
            "reverse_profit",
            "amount",
        ]
    ].to_numpy(
        dtype=float
    )

    number_of_people = len(values)

    rng = np.random.default_rng(
        seed
    )

    sampled_indices = rng.integers(
        low=0,
        high=number_of_people,
        size=(
            n_bootstrap,
            number_of_people,
        ),
    )

    sampled_totals = (
        values[
            sampled_indices
        ]
        .sum(axis=1)
    )

    samples = (
        sampled_totals[:, 0]
        / sampled_totals[:, 1]
    )

    return samples


def percentile_bootstrap_ci(
    samples,
    confidence=CONFIDENCE_LEVEL,
):
    """
    Calculate a two-sided percentile bootstrap interval.
    """
    samples = np.asarray(
        samples,
        dtype=float,
    )

    samples = samples[
        np.isfinite(samples)
    ]

    if len(samples) == 0:
        return np.nan, np.nan

    alpha = 1.0 - confidence

    lower, upper = np.quantile(
        samples,
        [
            alpha / 2.0,
            1.0 - alpha / 2.0,
        ],
    )

    return float(lower), float(upper)


def centered_bootstrap_positive_pvalue(
    observed,
    samples,
):
    """
    Calculate a one-sided, null-centered bootstrap p-value.

    Null hypothesis:
        lot-weighted RP/lot <= 0

    Alternative hypothesis:
        lot-weighted RP/lot > 0
    """
    samples = np.asarray(
        samples,
        dtype=float,
    )

    samples = samples[
        np.isfinite(samples)
    ]

    if (
        len(samples) == 0
        or not np.isfinite(observed)
    ):
        return np.nan

    # The ordinary bootstrap distribution is centered around
    # the observed edge. Subtracting the observed edge creates
    # a null distribution centered around zero.
    null_samples = (
        samples
        - observed
    )

    # Add one to the numerator and denominator to prevent a
    # simulated p-value of exactly zero.
    extreme_count = np.count_nonzero(
        null_samples >= observed
    )

    p_value = (
        extreme_count + 1
    ) / (
        len(null_samples) + 1
    )

    return float(p_value)


def holm_adjust(p_values):
    """
    Return step-down Holm-adjusted p-values for all
    registered strategy tests.
    """
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    if len(p_values) == 0:
        return np.array(
            [],
            dtype=float,
        )

    if not np.isfinite(
        p_values
    ).all():
        return np.full(
            len(p_values),
            np.nan,
            dtype=float,
        )

    if (
        (p_values < 0).any()
        or (p_values > 1).any()
    ):
        raise ValueError(
            "All p-values must lie between zero and one."
        )

    number_of_tests = len(
        p_values
    )

    order = np.argsort(
        p_values
    )

    adjusted = np.empty(
        number_of_tests,
        dtype=float,
    )

    running_maximum = 0.0

    for rank, original_index in enumerate(
        order
    ):
        remaining_tests = (
            number_of_tests
            - rank
        )

        candidate = min(
            1.0,
            remaining_tests
            * p_values[original_index],
        )

        running_maximum = max(
            running_maximum,
            candidate,
        )

        adjusted[
            original_index
        ] = running_maximum

    return adjusted


def evaluate_registered_strategies(
    decisions,
):
    """
    Evaluate coverage, lot-weighted edge, campaign stability,
    person-cluster uncertainty, and registered hypothesis tests.
    """
    required_columns = {
        "trade_row_id",
        "campaign_id",
        "person_id",
        "amount",
        "reverse_profit",
        "strategy",
        "threshold",
        "fade_decision",
    }

    missing_columns = (
        required_columns
        - set(decisions.columns)
    )

    if missing_columns:
        raise ValueError(
            "Decision table is missing columns: "
            f"{sorted(missing_columns)}"
        )

    summaries = []
    campaign_frames = []
    bootstrap_rows = []

    outcomes_available = (
        decisions[
            [
                "reverse_profit",
                "amount",
            ]
        ]
        .notna()
        .all()
        .all()
    )

    for strategy_name, strategy_data in decisions.groupby(
        "strategy",
        sort=False,
    ):
        selected = (
            strategy_data.loc[
                strategy_data[
                    "fade_decision"
                ]
            ]
            .copy()
        )

        trades_evaluated = len(
            strategy_data
        )

        trades_faded = len(
            selected
        )

        coverage = (
            trades_faded
            / trades_evaluated
            if trades_evaluated
            else np.nan
        )

        summary_row = {
            "strategy": strategy_name,
            "trades_evaluated": trades_evaluated,
            "trades_faded": trades_faded,
            "coverage": coverage,
        }

        if not outcomes_available:
            summary_row.update(
                {
                    "lot_weighted_rp_per_lot": np.nan,
                    "ci_lower": np.nan,
                    "ci_upper": np.nan,
                    "positive_campaigns": np.nan,
                    "total_campaigns": np.nan,
                    "positive_campaign_share": np.nan,
                    "p_value_one_sided": np.nan,
                }
            )

            summaries.append(
                summary_row
            )

            continue

        observed_edge = lot_weighted_edge(
            selected
        )

        bootstrap_samples = (
            person_cluster_bootstrap(
                selected,
                n_bootstrap=BOOTSTRAP_ITERATIONS,
                seed=SEED,
            )
        )

        ci_lower, ci_upper = (
            percentile_bootstrap_ci(
                bootstrap_samples,
                confidence=CONFIDENCE_LEVEL,
            )
        )

        p_value = (
            centered_bootstrap_positive_pvalue(
                observed=observed_edge,
                samples=bootstrap_samples,
            )
        )

        campaign_rows = []

        for campaign_id, campaign_data in (
            strategy_data.groupby(
                "campaign_id",
                sort=True,
            )
        ):
            faded = (
                campaign_data.loc[
                    campaign_data[
                        "fade_decision"
                    ]
                ]
                .copy()
            )

            campaign_thresholds = (
                campaign_data[
                    "threshold"
                ]
                .dropna()
                .unique()
            )

            if len(campaign_thresholds) != 1:
                raise AssertionError(
                    f"{strategy_name}, campaign {campaign_id}: "
                    "expected exactly one decision threshold."
                )

            campaign_rows.append(
                {
                    "strategy": strategy_name,
                    "campaign_id": campaign_id,
                    "trades_evaluated": len(
                        campaign_data
                    ),
                    "trades_faded": len(
                        faded
                    ),
                    "coverage": (
                        len(faded)
                        / len(campaign_data)
                    ),
                    "lot_weighted_rp_per_lot": (
                        lot_weighted_edge(
                            faded
                        )
                    ),
                    "decision_threshold": float(
                        campaign_thresholds[0]
                    ),
                }
            )

        campaign_frame = pd.DataFrame(
            campaign_rows
        )

        campaign_frames.append(
            campaign_frame
        )

        positive_campaigns = int(
            campaign_frame[
                "lot_weighted_rp_per_lot"
            ]
            .gt(0)
            .sum()
        )

        total_campaigns = len(
            campaign_frame
        )

        positive_campaign_share = (
            positive_campaigns
            / total_campaigns
            if total_campaigns
            else np.nan
        )

        summary_row.update(
            {
                "lot_weighted_rp_per_lot": observed_edge,
                "ci_lower": ci_lower,
                "ci_upper": ci_upper,
                "positive_campaigns": positive_campaigns,
                "total_campaigns": total_campaigns,
                "positive_campaign_share": (
                    positive_campaign_share
                ),
                "p_value_one_sided": p_value,
            }
        )

        summaries.append(
            summary_row
        )

        bootstrap_rows.extend(
            {
                "strategy": strategy_name,
                "iteration": iteration + 1,
                "lot_weighted_rp_per_lot": value,
            }
            for iteration, value
            in enumerate(
                bootstrap_samples
            )
        )

        print(
            f"  {strategy_name}: "
            f"faded={trades_faded:,}, "
            f"coverage={coverage:.2%}, "
            f"edge={observed_edge:.4f}, "
            f"{CONFIDENCE_LEVEL:.0%} CI="
            f"[{ci_lower:.4f}, {ci_upper:.4f}], "
            f"positive campaigns="
            f"{positive_campaigns}/{total_campaigns}"
        )

    evaluation_summary = pd.DataFrame(
        summaries
    )

    if outcomes_available:
        expected_strategy_count = len(
            REGISTERED_STRATEGIES
        )

        if len(evaluation_summary) != expected_strategy_count:
            raise AssertionError(
                "The number of evaluated strategies does not "
                "match the preregistered strategy count."
            )

        evaluation_summary[
            "holm_adjusted_p"
        ] = holm_adjust(
            evaluation_summary[
                "p_value_one_sided"
            ].to_numpy()
        )

        evaluation_summary[
            "holm_significant"
        ] = (
            evaluation_summary[
                "holm_adjusted_p"
            ]
            < 0.05
        )

        evaluation_summary[
            "edge_above_zero"
        ] = (
            evaluation_summary[
                "lot_weighted_rp_per_lot"
            ]
            > 0
        )

        evaluation_summary[
            "ci_clears_zero"
        ] = (
            evaluation_summary[
                "ci_lower"
            ]
            > 0
        )

        evaluation_summary[
            "campaign_share_pass"
        ] = (
            evaluation_summary[
                "positive_campaign_share"
            ]
            >= MIN_POSITIVE_CAMPAIGN_SHARE
        )

        evaluation_summary[
            "passes_all_registered_criteria"
        ] = (
            evaluation_summary[
                "edge_above_zero"
            ]
            & evaluation_summary[
                "ci_clears_zero"
            ]
            & evaluation_summary[
                "campaign_share_pass"
            ]
            & evaluation_summary[
                "holm_significant"
            ]
        )

    else:
        evaluation_summary[
            "holm_adjusted_p"
        ] = np.nan

        evaluation_summary[
            "holm_significant"
        ] = False

        evaluation_summary[
            "edge_above_zero"
        ] = False

        evaluation_summary[
            "ci_clears_zero"
        ] = False

        evaluation_summary[
            "campaign_share_pass"
        ] = False

        evaluation_summary[
            "passes_all_registered_criteria"
        ] = False

    campaign_output = (
        pd.concat(
            campaign_frames,
            ignore_index=True,
        )
        if campaign_frames
        else pd.DataFrame()
    )

    bootstrap_output = pd.DataFrame(
        bootstrap_rows
    )

    return (
        evaluation_summary,
        campaign_output,
        bootstrap_output,
        outcomes_available,
    )


# ======================================================
# Run registered evaluation
# ======================================================

if not RUN_SEALED_EVALUATION:
    print(
        "Skipped: sealed evaluation is locked."
    )

else:
    print(
        "Calculating registered metrics and "
        "Holm-adjusted inference..."
    )

    (
        evaluation_summary,
        campaign_results,
        bootstrap_results,
        outcomes_available,
    ) = evaluate_registered_strategies(
        sealed_decisions
    )

    display(
        evaluation_summary
    )

    if not outcomes_available:
        print(
            "Realized outcomes are unavailable. "
            "Only scores and decisions can be reported; "
            "performance evaluation was skipped."
        )

    else:
        primary_result = (
            evaluation_summary.loc[
                evaluation_summary[
                    "strategy"
                ].eq(
                    "walk_forward_gate"
                )
            ]
        )

        if len(primary_result) != 1:
            raise AssertionError(
                "Expected exactly one primary-strategy result."
            )

        primary_passed = bool(
            primary_result[
                "passes_all_registered_criteria"
            ].iloc[0]
        )

        print(
            "\nPrimary registered result:",
            "PASS"
            if primary_passed
            else "FAIL",
        )

Calculating registered metrics and Holm-adjusted inference...
  walk_forward_gate: faded=3,469, coverage=11.01%, edge=-2.4494, 95% CI=[-21.0491, 17.2068], positive campaigns=6/16
  original_continuity: faded=4,889, coverage=15.52%, edge=-4.2608, 95% CI=[-23.2614, 14.2531], positive campaigns=7/16


,strategy,trades_evaluated,trades_faded,coverage,lot_weighted_rp_per_lot,ci_lower,ci_upper,positive_campaigns,total_campaigns,positive_campaign_share,p_value_one_sided,holm_adjusted_p,holm_significant,edge_above_zero,ci_clears_zero,campaign_share_pass,passes_all_registered_criteria
0,walk_forward_gate,31498,3469,0.110134,-2.449398,-21.049081,17.206792,6,16,0.3750,0.596702,1.0,False,False,False,False,False
1,original_continuity,31498,4889,0.155216,-4.260824,-23.261351,14.253079,7,16,0.4375,0.666667,1.0,False,False,False,False,False



Primary registered result: FAIL


# Outputs

In [ ]:
if not RUN_SEALED_EVALUATION:
    print('No sealed output written. Preregistration and historical filter diagnostics are saved.')
else:
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
    # Never write personal email or IP addresses into research outputs.
    columns=['trade_row_id','campaign_id','strategy','score','threshold','stand_down','fade_decision']
    sealed_decisions[columns].to_csv(OUTPUT_DIR/'trade_decisions.csv',index=False)
    evaluation_summary.to_csv(OUTPUT_DIR/'evaluation_summary.csv',index=False)
    if outcomes_available:
        campaign_results.to_csv(OUTPUT_DIR/'campaign_performance.csv',index=False)
        bootstrap_results.to_csv(OUTPUT_DIR/'bootstrap_samples.csv',index=False)
    print('Saved registered outputs to',OUTPUT_DIR)
